In [1]:
from pathlib import Path
import pandas as pd

print("Pandas版本：", pd.__version__)
print("当前工作目录：", Path.cwd())

Pandas版本： 3.0.3
当前工作目录： /Users/nyx/Desktop/user-behavior-analysis/notebooks


In [2]:
DATA_PATH = Path("../data/raw/user_behavior_processed.csv")

print("数据文件是否存在：", DATA_PATH.exists())
print("数据文件完整路径：", DATA_PATH.resolve())
print("文件大小：", round(DATA_PATH.stat().st_size / 1024**2, 2), "MB")

数据文件是否存在： True
数据文件完整路径： /Users/nyx/Desktop/user-behavior-analysis/data/raw/user_behavior_processed.csv
文件大小： 469.46 MB


In [3]:
sample_df = pd.read_csv(
    DATA_PATH,
    nrows=10000
)

print("样本数据形状：", sample_df.shape)
display(sample_df.head())

样本数据形状： (10000, 5)


,time,user_id,item_id,item_category,behavior_type
0,2025-12-06 02,98047837,232431562,4245,1
1,2025-12-09 20,97726136,383583590,5894,1
2,2025-12-18 11,98607707,64749712,2883,1
3,2025-12-06 10,98662432,320593836,6562,1
4,2025-12-16 21,98145908,290208520,13926,1


In [4]:
print("字段名称：")
print(sample_df.columns.tolist())

print("\n各字段的数据类型：")
print(sample_df.dtypes)

print("\n数据基本信息：")
sample_df.info()

字段名称：
['time', 'user_id', 'item_id', 'item_category', 'behavior_type']

各字段的数据类型：
time               str
user_id          int64
item_id          int64
item_category    int64
behavior_type    int64
dtype: object

数据基本信息：
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   time           10000 non-null  str  
 1   user_id        10000 non-null  int64
 2   item_id        10000 non-null  int64
 3   item_category  10000 non-null  int64
 4   behavior_type  10000 non-null  int64
dtypes: int64(4), str(1)
memory usage: 517.7 KB


In [5]:
print("各字段缺失值数量：")
print(sample_df.isna().sum())

print("\n完全重复行数量：")
print(sample_df.duplicated().sum())

print("\n行为类型分布：")
print(sample_df["behavior_type"].value_counts().sort_index())

各字段缺失值数量：
time             0
user_id          0
item_id          0
item_category    0
behavior_type    0
dtype: int64

完全重复行数量：
717

行为类型分布：
behavior_type
1    9430
2     195
3     276
4      99
Name: count, dtype: int64


In [6]:
sample_df["time"] = pd.to_datetime(
    sample_df["time"],
    format="%Y-%m-%d %H",
    errors="coerce"
)

print("转换后的字段类型：")
print(sample_df.dtypes)

print("\n无法转换的时间数量：")
print(sample_df["time"].isna().sum())

print("\n最早时间：", sample_df["time"].min())
print("最晚时间：", sample_df["time"].max())

转换后的字段类型：
time             datetime64[us]
user_id                   int64
item_id                   int64
item_category             int64
behavior_type             int64
dtype: object

无法转换的时间数量：
0

最早时间： 2025-11-18 00:00:00
最晚时间： 2025-12-18 23:00:00


In [7]:
duplicate_rows = sample_df[
    sample_df.duplicated(keep=False)
].sort_values(
    ["user_id", "item_id", "time", "behavior_type"]
)

print("重复记录数量：", len(duplicate_rows))
display(duplicate_rows.head(20))

重复记录数量： 1395


,time,user_id,item_id,item_category,behavior_type
9845,2025-12-03 11:00:00,2967122,17123536,11178,1
9854,2025-12-03 11:00:00,2967122,17123536,11178,1
9467,2025-12-03 16:00:00,2967122,176322914,2866,1
9503,2025-12-03 16:00:00,2967122,176322914,2866,1
9431,2025-11-22 05:00:00,2967122,272639944,5027,1
9476,2025-11-22 05:00:00,2967122,272639944,5027,1
9611,2025-11-21 15:00:00,2967122,273419428,169,1
9647,2025-11-21 15:00:00,2967122,273419428,169,1
2530,2025-11-29 21:00:00,10131505,66223566,4677,1
2908,2025-11-29 21:00:00,10131505,66223566,4677,1


In [8]:
behavior_mapping = {
    1: "浏览",
    2: "收藏",
    3: "加购",
    4: "购买"
}

behavior_summary = (
    sample_df["behavior_type"]
    .value_counts()
    .sort_index()
    .rename_axis("behavior_type")
    .reset_index(name="count")
)

behavior_summary["behavior_name"] = behavior_summary["behavior_type"].map(
    behavior_mapping
)

behavior_summary["percentage"] = (
    behavior_summary["count"] / len(sample_df) * 100
).round(2)

display(behavior_summary)

,behavior_type,count,behavior_name,percentage
0,1,9430,浏览,94.30
1,2,195,收藏,1.95
2,3,276,加购,2.76
3,4,99,购买,0.99


In [9]:
dtype_mapping = {
    "user_id": "int64",
    "item_id": "int64",
    "item_category": "int32",
    "behavior_type": "int8"
}

chunk_size = 500_000

chunk_reader = pd.read_csv(
    DATA_PATH,
    dtype=dtype_mapping,
    parse_dates=["time"],
    chunksize=chunk_size
)

print("分块读取器创建成功")



分块读取器创建成功


In [10]:
total_rows = 0
missing_counts = None
behavior_counts = {}
min_time = None
max_time = None
chunk_count = 0

for chunk in pd.read_csv(
    DATA_PATH,
    dtype=dtype_mapping,
    parse_dates=["time"],
    chunksize=chunk_size
):
    chunk_count += 1
    total_rows += len(chunk)

    # 缺失值统计
    current_missing = chunk.isna().sum()

    if missing_counts is None:
        missing_counts = current_missing
    else:
        missing_counts = missing_counts.add(
            current_missing,
            fill_value=0
        )

    # 行为类型统计
    current_behavior = chunk["behavior_type"].value_counts()

    for behavior_type, count in current_behavior.items():
        behavior_counts[behavior_type] = (
            behavior_counts.get(behavior_type, 0) + count
        )

    # 时间范围
    current_min_time = chunk["time"].min()
    current_max_time = chunk["time"].max()

    if min_time is None or current_min_time < min_time:
        min_time = current_min_time

    if max_time is None or current_max_time > max_time:
        max_time = current_max_time

    print(f"已完成第 {chunk_count} 个数据块，累计 {total_rows:,} 行")

    

已完成第 1 个数据块，累计 500,000 行
已完成第 2 个数据块，累计 1,000,000 行
已完成第 3 个数据块，累计 1,500,000 行
已完成第 4 个数据块，累计 2,000,000 行
已完成第 5 个数据块，累计 2,500,000 行
已完成第 6 个数据块，累计 3,000,000 行
已完成第 7 个数据块，累计 3,500,000 行
已完成第 8 个数据块，累计 4,000,000 行
已完成第 9 个数据块，累计 4,500,000 行
已完成第 10 个数据块，累计 5,000,000 行
已完成第 11 个数据块，累计 5,500,000 行
已完成第 12 个数据块，累计 6,000,000 行
已完成第 13 个数据块，累计 6,500,000 行
已完成第 14 个数据块，累计 7,000,000 行
已完成第 15 个数据块，累计 7,500,000 行
已完成第 16 个数据块，累计 8,000,000 行
已完成第 17 个数据块，累计 8,500,000 行
已完成第 18 个数据块，累计 9,000,000 行
已完成第 19 个数据块，累计 9,500,000 行
已完成第 20 个数据块，累计 10,000,000 行
已完成第 21 个数据块，累计 10,500,000 行
已完成第 22 个数据块，累计 11,000,000 行
已完成第 23 个数据块，累计 11,500,000 行
已完成第 24 个数据块，累计 12,000,000 行
已完成第 25 个数据块，累计 12,256,906 行


In [11]:
print("完整数据总行数：", f"{total_rows:,}")
print("数据块数量：", chunk_count)

print("\n各字段缺失值：")
print(missing_counts.astype("int64"))

print("\n行为类型数量：")
for behavior_type in sorted(behavior_counts):
    print(
        behavior_type,
        behavior_mapping[behavior_type],
        f"{behavior_counts[behavior_type]:,}"
    )

print("\n最早时间：", min_time)
print("最晚时间：", max_time)

完整数据总行数： 12,256,906
数据块数量： 25

各字段缺失值：
time             0
user_id          0
item_id          0
item_category    0
behavior_type    0
dtype: int64

行为类型数量：
1 浏览 11,550,581
2 收藏 242,556
3 加购 343,564
4 购买 120,205

最早时间： 2025-11-18 00:00:00
最晚时间： 2025-12-18 23:00:00


In [12]:
full_behavior_summary = pd.DataFrame(
    {
        "behavior_type": list(behavior_counts.keys()),
        "count": list(behavior_counts.values())
    }
).sort_values("behavior_type")

full_behavior_summary["behavior_name"] = (
    full_behavior_summary["behavior_type"]
    .map(behavior_mapping)
)

full_behavior_summary["percentage"] = (
    full_behavior_summary["count"]
    / total_rows
    * 100
).round(2)

display(full_behavior_summary)


,behavior_type,count,behavior_name,percentage
0,1,11550581,浏览,94.24
2,2,242556,收藏,1.98
1,3,343564,加购,2.80
3,4,120205,购买,0.98


In [13]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("../data/raw/user_behavior_processed.csv")

dtype_mapping = {
    "user_id": "int64",
    "item_id": "int64",
    "item_category": "int32",
    "behavior_type": "int8"
}

df = pd.read_csv(
    DATA_PATH,
    dtype=dtype_mapping,
    parse_dates=["time"],
    memory_map=True
)

print("全部数据读取完成")
print("数据行列数：", df.shape)
print("总行数：", f"{len(df):,}")
print("总列数：", df.shape[1])
print(
    "内存占用：",
    round(df.memory_usage(deep=True).sum() / 1024**2, 2),
    "MB"
)



全部数据读取完成
数据行列数： (12256906, 5)
总行数： 12,256,906
总列数： 5
内存占用： 338.98 MB


In [14]:
print(df.shape)

(12256906, 5)


In [15]:
from pathlib import Path
from collections import Counter
import pandas as pd

# Notebook 位于 notebooks 文件夹时使用这个路径
DATA_PATH = Path("../data/raw/user_behavior_processed.csv")
CHUNK_SIZE = 500_000

total_rows = 0
user_ids = set()
item_ids = set()
category_ids = set()

behavior_counts = Counter()
behavior_users = {1: set(), 2: set(), 3: set(), 4: set()}

min_time = None
max_time = None
missing_counts = Counter()

columns = [
    "time",
    "user_id",
    "item_id",
    "item_category",
    "behavior_type"
]

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=columns,
    chunksize=CHUNK_SIZE
):
    total_rows += len(chunk)

    # 去重主体数量
    user_ids.update(chunk["user_id"].dropna().unique())
    item_ids.update(chunk["item_id"].dropna().unique())
    category_ids.update(chunk["item_category"].dropna().unique())

    # 缺失值统计
    missing_counts.update(chunk.isna().sum().to_dict())

    # 四种行为统计
    behavior = pd.to_numeric(
        chunk["behavior_type"],
        errors="coerce"
    )

    valid_behavior = behavior.dropna().astype(int)
    behavior_counts.update(valid_behavior.tolist())

    for behavior_code in [1, 2, 3, 4]:
        users = chunk.loc[
            behavior == behavior_code, "user_id"
        ].dropna().unique()

        behavior_users[behavior_code].update(users)

    # 时间范围
    parsed_time = pd.to_datetime(
        chunk["time"],
        errors="coerce"
    ).dropna()

    if not parsed_time.empty:
        chunk_min = parsed_time.min()
        chunk_max = parsed_time.max()

        min_time = (
            chunk_min
            if min_time is None
            else min(min_time, chunk_min)
        )

        max_time = (
            chunk_max
            if max_time is None
            else max(max_time, chunk_max)
        )

# 数据概况表
overview = pd.DataFrame({
    "指标": [
        "行为记录总数",
        "去重用户数",
        "去重商品数",
        "商品品类数",
        "开始时间",
        "结束时间"
    ],
    "结果": [
        f"{total_rows:,}",
        f"{len(user_ids):,}",
        f"{len(item_ids):,}",
        f"{len(category_ids):,}",
        min_time,
        max_time
    ]
})

behavior_names = {
    1: "浏览",
    2: "收藏",
    3: "加购",
    4: "购买"
}

behavior_table = pd.DataFrame([
    {
        "行为类型": behavior_names[code],
        "行为次数": behavior_counts[code],
        "行为占比": (
            behavior_counts[code] / total_rows
            if total_rows else 0
        ),
        "涉及用户数": len(behavior_users[code])
    }
    for code in [1, 2, 3, 4]
])

behavior_table["行为次数"] = behavior_table["行为次数"].map(
    lambda x: f"{x:,}"
)
behavior_table["涉及用户数"] = behavior_table["涉及用户数"].map(
    lambda x: f"{x:,}"
)
behavior_table["行为占比"] = behavior_table["行为占比"].map(
    lambda x: f"{x:.2%}"
)

print("一、数据基础概况")
display(overview)

print("\n二、用户行为分布")
display(behavior_table)

print("\n三、各字段缺失值")
display(pd.DataFrame({
    "字段": columns,
    "缺失数量": [missing_counts[col] for col in columns]
}))


一、数据基础概况


,指标,结果
0,行为记录总数,"12,256,906"
1,去重用户数,"10,000"
2,去重商品数,"2,876,947"
3,商品品类数,"8,916"
4,开始时间,2025-11-18 00:00:00
5,结束时间,2025-12-18 23:00:00



二、用户行为分布


,行为类型,行为次数,行为占比,涉及用户数
0,浏览,"11,550,581",94.24%,"10,000"
1,收藏,"242,556",1.98%,"6,730"
2,加购,"343,564",2.80%,"8,614"
3,购买,"120,205",0.98%,"8,886"



三、各字段缺失值


,字段,缺失数量
0,time,0
1,user_id,0
2,item_id,0
3,item_category,0
4,behavior_type,0


In [16]:
from pathlib import Path
from collections import Counter
import pandas as pd

# 自动寻找原始数据
possible_paths = [
    Path("../data/raw/user_behavior_processed.csv"),
    Path("data/raw/user_behavior_processed.csv")
]

DATA_PATH = next(
    (path for path in possible_paths if path.exists()),
    None
)

if DATA_PATH is None:
    raise FileNotFoundError("没有找到 user_behavior_processed.csv")

print("读取文件：", DATA_PATH)

columns = [
    "time",
    "user_id",
    "item_id",
    "item_category",
    "behavior_type"
]

total_rows = 0
users = set()
items = set()
categories = set()
behavior_counts = Counter()

start_time = None
end_time = None

# 分块读取，不会一次性占满内存
for chunk in pd.read_csv(
    DATA_PATH,
    usecols=columns,
    chunksize=500_000
):
    total_rows += len(chunk)

    # 这里只是统计唯一数量，不是删除重复行为
    users.update(chunk["user_id"].dropna().unique())
    items.update(chunk["item_id"].dropna().unique())
    categories.update(chunk["item_category"].dropna().unique())

    behavior = pd.to_numeric(
        chunk["behavior_type"],
        errors="coerce"
    ).dropna().astype(int)

    behavior_counts.update(behavior.tolist())

    times = pd.to_datetime(
        chunk["time"],
        errors="coerce"
    ).dropna()

    if not times.empty:
        chunk_start = times.min()
        chunk_end = times.max()

        start_time = (
            chunk_start
            if start_time is None
            else min(start_time, chunk_start)
        )

        end_time = (
            chunk_end
            if end_time is None
            else max(end_time, chunk_end)
        )

print("\n===== 原始数据基础统计 =====")
print(f"行为记录总数：{total_rows:,}")
print(f"用户数量：{len(users):,}")
print(f"商品数量：{len(items):,}")
print(f"商品品类数量：{len(categories):,}")
print(f"时间范围：{start_time} 至 {end_time}")

print("\n===== 行为数量 =====")
print(f"浏览：{behavior_counts[1]:,}")
print(f"收藏：{behavior_counts[2]:,}")
print(f"加购：{behavior_counts[3]:,}")
print(f"购买：{behavior_counts[4]:,}")

读取文件： ../data/raw/user_behavior_processed.csv

===== 原始数据基础统计 =====
行为记录总数：12,256,906
用户数量：10,000
商品数量：2,876,947
商品品类数量：8,916
时间范围：2025-11-18 00:00:00 至 2025-12-18 23:00:00

===== 行为数量 =====
浏览：11,550,581
收藏：242,556
加购：343,564
购买：120,205


In [17]:
import pandas as pd

columns = [
    "time",
    "user_id",
    "item_id",
    "item_category",
    "behavior_type"
]

missing_counts = pd.Series(
    0,
    index=columns,
    dtype="int64"
)

invalid_id_counts = {
    "user_id": 0,
    "item_id": 0,
    "item_category": 0
}

invalid_behavior_count = 0
invalid_time_count = 0
problem_row_count = 0
checked_rows = 0

for chunk_number, chunk in enumerate(
    pd.read_csv(
        DATA_PATH,
        usecols=columns,
        chunksize=500_000
    ),
    start=1
):
    checked_rows += len(chunk)

    # 1. 检查缺失值
    missing_counts += chunk.isna().sum()

    # 标记本批数据中存在问题的行
    problem_mask = chunk.isna().any(axis=1)

    # 2. 检查用户、商品和品类ID
    for column in [
        "user_id",
        "item_id",
        "item_category"
    ]:
        numeric_values = pd.to_numeric(
            chunk[column],
            errors="coerce"
        )

        invalid_id_mask = (
            chunk[column].notna()
            & (
                numeric_values.isna()
                | (numeric_values <= 0)
            )
        )

        invalid_id_counts[column] += int(
            invalid_id_mask.sum()
        )

        problem_mask |= invalid_id_mask

    # 3. 检查行为编码是否属于1、2、3、4
    behavior_values = pd.to_numeric(
        chunk["behavior_type"],
        errors="coerce"
    )

    invalid_behavior_mask = (
        chunk["behavior_type"].notna()
        & ~behavior_values.isin([1, 2, 3, 4])
    )

    invalid_behavior_count += int(
        invalid_behavior_mask.sum()
    )

    problem_mask |= invalid_behavior_mask

    # 4. 检查时间能否正常解析
    parsed_time = pd.to_datetime(
        chunk["time"],
        errors="coerce"
    )

    invalid_time_mask = (
        chunk["time"].notna()
        & parsed_time.isna()
    )

    invalid_time_count += int(
        invalid_time_mask.sum()
    )

    problem_mask |= invalid_time_mask

    # 存在至少一个问题的行数
    problem_row_count += int(problem_mask.sum())

    print(
        f"已检查第 {chunk_number} 个数据块，"
        f"累计 {checked_rows:,} 行"
    )

print("\n===== 数据质量检查结果 =====")

quality_table = pd.DataFrame({
    "检查项目": [
        "总记录数",
        "存在至少一个问题的记录数",
        "user_id非法值",
        "item_id非法值",
        "item_category非法值",
        "behavior_type非法值",
        "time无法解析"
    ],
    "数量": [
        checked_rows,
        problem_row_count,
        invalid_id_counts["user_id"],
        invalid_id_counts["item_id"],
        invalid_id_counts["item_category"],
        invalid_behavior_count,
        invalid_time_count
    ]
})

display(quality_table)

print("\n===== 各字段缺失值 =====")

missing_table = pd.DataFrame({
    "字段": columns,
    "缺失数量": [
        int(missing_counts[column])
        for column in columns
    ],
    "缺失率": [
        f"{missing_counts[column] / checked_rows:.4%}"
        for column in columns
    ]
})

display(missing_table)

已检查第 1 个数据块，累计 500,000 行
已检查第 2 个数据块，累计 1,000,000 行
已检查第 3 个数据块，累计 1,500,000 行
已检查第 4 个数据块，累计 2,000,000 行
已检查第 5 个数据块，累计 2,500,000 行
已检查第 6 个数据块，累计 3,000,000 行
已检查第 7 个数据块，累计 3,500,000 行
已检查第 8 个数据块，累计 4,000,000 行
已检查第 9 个数据块，累计 4,500,000 行
已检查第 10 个数据块，累计 5,000,000 行
已检查第 11 个数据块，累计 5,500,000 行
已检查第 12 个数据块，累计 6,000,000 行
已检查第 13 个数据块，累计 6,500,000 行
已检查第 14 个数据块，累计 7,000,000 行
已检查第 15 个数据块，累计 7,500,000 行
已检查第 16 个数据块，累计 8,000,000 行
已检查第 17 个数据块，累计 8,500,000 行
已检查第 18 个数据块，累计 9,000,000 行
已检查第 19 个数据块，累计 9,500,000 行
已检查第 20 个数据块，累计 10,000,000 行
已检查第 21 个数据块，累计 10,500,000 行
已检查第 22 个数据块，累计 11,000,000 行
已检查第 23 个数据块，累计 11,500,000 行
已检查第 24 个数据块，累计 12,000,000 行
已检查第 25 个数据块，累计 12,256,906 行

===== 数据质量检查结果 =====


,检查项目,数量
0,总记录数,12256906
1,存在至少一个问题的记录数,0
2,user_id非法值,0
3,item_id非法值,0
4,item_category非法值,0
5,behavior_type非法值,0
6,time无法解析,0



===== 各字段缺失值 =====


,字段,缺失数量,缺失率
0,time,0,0.0000%
1,user_id,0,0.0000%
2,item_id,0,0.0000%
3,item_category,0,0.0000%
4,behavior_type,0,0.0000%


In [18]:
import pandas as pd

numeric_columns = [
    "user_id",
    "item_id",
    "item_category",
    "behavior_type"
]

value_ranges = {
    column: {
        "最小值": None,
        "最大值": None
    }
    for column in numeric_columns
}

for chunk_number, chunk in enumerate(
    pd.read_csv(
        DATA_PATH,
        usecols=numeric_columns,
        chunksize=500_000
    ),
    start=1
):
    for column in numeric_columns:
        values = pd.to_numeric(
            chunk[column],
            errors="raise"
        )

        current_min = int(values.min())
        current_max = int(values.max())

        old_min = value_ranges[column]["最小值"]
        old_max = value_ranges[column]["最大值"]

        value_ranges[column]["最小值"] = (
            current_min
            if old_min is None
            else min(old_min, current_min)
        )

        value_ranges[column]["最大值"] = (
            current_max
            if old_max is None
            else max(old_max, current_max)
        )

    print(f"已检查第 {chunk_number} 个数据块")

def recommend_dtype(min_value, max_value):
    if min_value >= 0:
        if max_value <= 255:
            return "uint8"
        elif max_value <= 65_535:
            return "uint16"
        elif max_value <= 4_294_967_295:
            return "uint32"
        else:
            return "uint64"
    else:
        if -128 <= min_value and max_value <= 127:
            return "int8"
        elif -32_768 <= min_value and max_value <= 32_767:
            return "int16"
        elif (
            -2_147_483_648 <= min_value
            and max_value <= 2_147_483_647
        ):
            return "int32"
        else:
            return "int64"

range_table = pd.DataFrame([
    {
        "字段": column,
        "最小值": value_ranges[column]["最小值"],
        "最大值": value_ranges[column]["最大值"],
        "建议类型": recommend_dtype(
            value_ranges[column]["最小值"],
            value_ranges[column]["最大值"]
        )
    }
    for column in numeric_columns
])

print("\n===== 字段范围与建议类型 =====")
display(range_table)


已检查第 1 个数据块
已检查第 2 个数据块
已检查第 3 个数据块
已检查第 4 个数据块
已检查第 5 个数据块
已检查第 6 个数据块
已检查第 7 个数据块
已检查第 8 个数据块
已检查第 9 个数据块
已检查第 10 个数据块
已检查第 11 个数据块
已检查第 12 个数据块
已检查第 13 个数据块
已检查第 14 个数据块
已检查第 15 个数据块
已检查第 16 个数据块
已检查第 17 个数据块
已检查第 18 个数据块
已检查第 19 个数据块
已检查第 20 个数据块
已检查第 21 个数据块
已检查第 22 个数据块
已检查第 23 个数据块
已检查第 24 个数据块
已检查第 25 个数据块

===== 字段范围与建议类型 =====


,字段,最小值,最大值,建议类型
0,user_id,4913,142455899,uint32
1,item_id,64,404562461,uint32
2,item_category,2,14080,uint16
3,behavior_type,1,4,uint8


In [19]:
from pathlib import Path
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

columns = [
    "time",
    "user_id",
    "item_id",
    "item_category",
    "behavior_type"
]

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

# 避免覆盖之前生成的文件
version = 1
OUTPUT_PATH = (
    processed_dir
    / f"user_behavior_standardized_v{version}.parquet"
)

while OUTPUT_PATH.exists():
    version += 1
    OUTPUT_PATH = (
        processed_dir
        / f"user_behavior_standardized_v{version}.parquet"
    )

writer = None
written_rows = 0

try:
    for chunk_number, chunk in enumerate(
        pd.read_csv(
            DATA_PATH,
            usecols=columns,
            chunksize=500_000
        ),
        start=1
    ):
        # 统一时间格式
        chunk["time"] = pd.to_datetime(
            chunk["time"],
            errors="raise"
        )

        # 压缩字段类型
        chunk["user_id"] = chunk["user_id"].astype(
            "uint32"
        )
        chunk["item_id"] = chunk["item_id"].astype(
            "uint32"
        )
        chunk["item_category"] = chunk[
            "item_category"
        ].astype("uint16")
        chunk["behavior_type"] = chunk[
            "behavior_type"
        ].astype("uint8")

        # 固定字段顺序
        chunk = chunk[columns]

        table = pa.Table.from_pandas(
            chunk,
            preserve_index=False
        )

        if writer is None:
            writer = pq.ParquetWriter(
                OUTPUT_PATH,
                table.schema,
                compression="snappy"
            )

        writer.write_table(table)
        written_rows += len(chunk)

        print(
            f"已写入第 {chunk_number} 个数据块，"
            f"累计 {written_rows:,} 行"
        )

finally:
    if writer is not None:
        writer.close()

# 验证生成结果
parquet_file = pq.ParquetFile(OUTPUT_PATH)
verified_rows = parquet_file.metadata.num_rows

csv_size_mb = DATA_PATH.stat().st_size / 1024**2
parquet_size_mb = OUTPUT_PATH.stat().st_size / 1024**2

print("\n===== 标准化数据生成结果 =====")
print("输出文件：", OUTPUT_PATH)
print(f"写入记录数：{written_rows:,}")
print(f"Parquet验证行数：{verified_rows:,}")
print(f"原始CSV大小：{csv_size_mb:.2f} MB")
print(f"Parquet大小：{parquet_size_mb:.2f} MB")
print(f"存储空间减少：{1 - parquet_size_mb / csv_size_mb:.2%}")

print("\n===== Parquet字段结构 =====")
print(parquet_file.schema_arrow)

已写入第 1 个数据块，累计 500,000 行
已写入第 2 个数据块，累计 1,000,000 行
已写入第 3 个数据块，累计 1,500,000 行
已写入第 4 个数据块，累计 2,000,000 行
已写入第 5 个数据块，累计 2,500,000 行
已写入第 6 个数据块，累计 3,000,000 行
已写入第 7 个数据块，累计 3,500,000 行
已写入第 8 个数据块，累计 4,000,000 行
已写入第 9 个数据块，累计 4,500,000 行
已写入第 10 个数据块，累计 5,000,000 行
已写入第 11 个数据块，累计 5,500,000 行
已写入第 12 个数据块，累计 6,000,000 行
已写入第 13 个数据块，累计 6,500,000 行
已写入第 14 个数据块，累计 7,000,000 行
已写入第 15 个数据块，累计 7,500,000 行
已写入第 16 个数据块，累计 8,000,000 行
已写入第 17 个数据块，累计 8,500,000 行
已写入第 18 个数据块，累计 9,000,000 行
已写入第 19 个数据块，累计 9,500,000 行
已写入第 20 个数据块，累计 10,000,000 行
已写入第 21 个数据块，累计 10,500,000 行
已写入第 22 个数据块，累计 11,000,000 行
已写入第 23 个数据块，累计 11,500,000 行
已写入第 24 个数据块，累计 12,000,000 行
已写入第 25 个数据块，累计 12,256,906 行

===== 标准化数据生成结果 =====
输出文件： ../data/processed/user_behavior_standardized_v1.parquet
写入记录数：12,256,906
Parquet验证行数：12,256,906
原始CSV大小：469.46 MB
Parquet大小：94.38 MB
存储空间减少：79.90%

===== Parquet字段结构 =====
time: timestamp[us]
user_id: uint32
item_id: uint32
item_category: uint16
behavior_type: uint8
-- schema

In [20]:
from pathlib import Path

reports_dir = Path("../reports")
reports_dir.mkdir(parents=True, exist_ok=True)

REPORT_PATH = reports_dir / "data_quality_report.md"

behavior_names = {
    1: "浏览",
    2: "收藏",
    3: "加购",
    4: "购买"
}

behavior_lines = []

for code in [1, 2, 3, 4]:
    count = behavior_counts[code]
    ratio = count / total_rows

    behavior_lines.append(
        f"| {code} | {behavior_names[code]} | "
        f"{count:,} | {ratio:.2%} |"
    )

behavior_table_text = "\n".join(behavior_lines)

report_text = f"""# 淘宝用户行为数据质量校验报告

## 1. 数据基础概况

| 指标 | 统计结果 |
|---|---:|
| 行为记录总数 | {total_rows:,} |
| 用户数量 | {len(users):,} |
| 商品数量 | {len(items):,} |
| 商品品类数量 | {len(categories):,} |
| 开始时间 | {start_time} |
| 结束时间 | {end_time} |

## 2. 用户行为分布

| 行为编码 | 行为类型 | 行为次数 | 行为占比 |
|---:|---|---:|---:|
{behavior_table_text}

说明：行为占比反映各类行为记录在全部事件中的比例，
不直接等同于用户转化率。

## 3. 数据完整性检查

| 检查项目 | 结果 |
|---|---:|
| 存在至少一个问题的记录数 | {problem_row_count:,} |
| time缺失值 | {int(missing_counts["time"]):,} |
| user_id缺失值 | {int(missing_counts["user_id"]):,} |
| item_id缺失值 | {int(missing_counts["item_id"]):,} |
| item_category缺失值 | {int(missing_counts["item_category"]):,} |
| behavior_type缺失值 | {int(missing_counts["behavior_type"]):,} |
| user_id非法值 | {invalid_id_counts["user_id"]:,} |
| item_id非法值 | {invalid_id_counts["item_id"]:,} |
| item_category非法值 | {invalid_id_counts["item_category"]:,} |
| behavior_type非法值 | {invalid_behavior_count:,} |
| 无法解析的时间记录 | {invalid_time_count:,} |

## 4. 字段标准化结果

| 字段 | 标准化类型 |
|---|---|
| time | timestamp |
| user_id | uint32 |
| item_id | uint32 |
| item_category | uint16 |
| behavior_type | uint8 |

## 5. 重复行为处理原则

本阶段未直接删除任何重复行为记录。

同一用户对同一商品产生的重复浏览、收藏、加购和购买行为
具有实际业务意义，可以反映用户兴趣强度、购买意愿、
决策犹豫程度和复购行为。

后续特征工程将基于这些记录构建：

- 重复浏览次数；
- 重复收藏次数；
- 重复加购次数；
- 重复购买次数；
- 行为频率；
- 行为时间间隔。

## 6. 存储优化结果

| 指标 | 结果 |
|---|---:|
| 原始CSV大小 | {csv_size_mb:.2f} MB |
| Parquet文件大小 | {parquet_size_mb:.2f} MB |
| 存储空间减少 | {1 - parquet_size_mb / csv_size_mb:.2%} |
| Parquet记录数 | {verified_rows:,} |

## 7. 校验结论

原始数据共包含{total_rows:,}条用户行为记录。
所有核心字段均不存在缺失值或非法值，时间字段能够正常解析，
四类行为记录之和与数据总行数一致。

标准化处理完整保留了原始行为记录，并通过字段类型压缩和
Parquet列式存储将文件体积减少
{1 - parquet_size_mb / csv_size_mb:.2%}。

注意：数据文件的时间范围显示为2025年11月18日至12月18日，
最终使用前需要根据数据来源说明确认日期是否经过匿名化或平移。
"""

REPORT_PATH.write_text(
    report_text,
    encoding="utf-8"
)

print("数据质量报告已生成：")
print(REPORT_PATH)
print("\n===== 报告预览 =====\n")
print(report_text)

数据质量报告已生成：
../reports/data_quality_report.md

===== 报告预览 =====

# 淘宝用户行为数据质量校验报告

## 1. 数据基础概况

| 指标 | 统计结果 |
|---|---:|
| 行为记录总数 | 12,256,906 |
| 用户数量 | 10,000 |
| 商品数量 | 2,876,947 |
| 商品品类数量 | 8,916 |
| 开始时间 | 2025-11-18 00:00:00 |
| 结束时间 | 2025-12-18 23:00:00 |

## 2. 用户行为分布

| 行为编码 | 行为类型 | 行为次数 | 行为占比 |
|---:|---|---:|---:|
| 1 | 浏览 | 11,550,581 | 94.24% |
| 2 | 收藏 | 242,556 | 1.98% |
| 3 | 加购 | 343,564 | 2.80% |
| 4 | 购买 | 120,205 | 0.98% |

说明：行为占比反映各类行为记录在全部事件中的比例，
不直接等同于用户转化率。

## 3. 数据完整性检查

| 检查项目 | 结果 |
|---|---:|
| 存在至少一个问题的记录数 | 0 |
| time缺失值 | 0 |
| user_id缺失值 | 0 |
| item_id缺失值 | 0 |
| item_category缺失值 | 0 |
| behavior_type缺失值 | 0 |
| user_id非法值 | 0 |
| item_id非法值 | 0 |
| item_category非法值 | 0 |
| behavior_type非法值 | 0 |
| 无法解析的时间记录 | 0 |

## 4. 字段标准化结果

| 字段 | 标准化类型 |
|---|---|
| time | timestamp |
| user_id | uint32 |
| item_id | uint32 |
| item_category | uint16 |
| behavior_type | uint8 |

## 5. 重复行为处理原则

本阶段未直接删除任何重复行为记录。

同一用户对同一商品产生的重复浏览、收藏、加购和购买行为
具有实际业务意义，可以反映用户兴

In [21]:
from pathlib import Path
import subprocess

PROJECT_ROOT = Path("..").resolve()
GITIGNORE_PATH = PROJECT_ROOT / ".gitignore"

print("===== 当前 .gitignore =====")

if GITIGNORE_PATH.exists():
    print(GITIGNORE_PATH.read_text(encoding="utf-8"))
else:
    print("没有找到 .gitignore")

print("\n===== 当前 Git 状态 =====")

result = subprocess.run(
    ["git", "status", "--short"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True
)

print(result.stdout or "当前没有未提交的改动")

===== 当前 .gitignore =====
.venv/
__pycache__/
*.pyc
.DS_Store

.ipynb_checkpoints/
**/.ipynb_checkpoints/

data/raw/*
!data/raw/.gitkeep

data/processed/*
!data/processed/.gitkeep

outputs/*
!outputs/.gitkeep

*.db


===== 当前 Git 状态 =====
 M notebooks/01_data_loading.ipynb
?? reports/data_quality_report.md



In [22]:
import pyarrow.parquet as pq

parquet_path = "../data/processed/user_behavior_standardized_v1.parquet"

preview = (
    pq.ParquetFile(parquet_path)
    .read_row_group(0)
    .slice(0, 10)
    .to_pandas()
)

display(preview)



,time,user_id,item_id,item_category,behavior_type
0,2025-12-06 02:00:00,98047837,232431562,4245,1
1,2025-12-09 20:00:00,97726136,383583590,5894,1
2,2025-12-18 11:00:00,98607707,64749712,2883,1
3,2025-12-06 10:00:00,98662432,320593836,6562,1
4,2025-12-16 21:00:00,98145908,290208520,13926,1
5,2025-12-03 20:00:00,93784494,337869048,3979,1
6,2025-12-13 20:00:00,94832743,105749725,9559,1
7,2025-11-27 16:00:00,95290487,76866650,10875,1
8,2025-12-11 23:00:00,96610296,161166643,3064,1
9,2025-12-05 23:00:00,100684618,21751142,2158,3


In [24]:
%pip install duckdb

Note: you may need to restart the kernel to use updated packages.


In [25]:
from pathlib import Path
import duckdb

PARQUET_PATH = Path(
    "../data/processed/"
    "user_behavior_standardized_v1.parquet"
).resolve()

DATABASE_PATH = Path(
    "../data/processed/"
    "user_behavior_analytics.db"
).resolve()

# 建立DuckDB数据库连接
con = duckdb.connect(str(DATABASE_PATH))

parquet_sql_path = PARQUET_PATH.as_posix()

# 创建用户维度中间表
con.execute(f"""
CREATE OR REPLACE TABLE user_summary AS
SELECT
    user_id,

    COUNT(*) AS total_actions,

    COUNT(*) FILTER (
        WHERE behavior_type = 1
    ) AS view_count,

    COUNT(*) FILTER (
        WHERE behavior_type = 2
    ) AS favorite_count,

    COUNT(*) FILTER (
        WHERE behavior_type = 3
    ) AS cart_count,

    COUNT(*) FILTER (
        WHERE behavior_type = 4
    ) AS purchase_count,

    COUNT(DISTINCT item_id) AS unique_items,

    COUNT(DISTINCT item_category)
        AS unique_categories,

    COUNT(DISTINCT CAST(time AS DATE))
        AS active_days,

    MIN(time) AS first_action_time,

    MAX(time) AS last_action_time,

    COUNT(DISTINCT CASE
        WHEN behavior_type = 4
        THEN item_id
    END) AS purchased_items,

    ROUND(
        100.0
        * COUNT(*) FILTER (
            WHERE behavior_type = 4
        )
        / COUNT(*),
        4
    ) AS purchase_action_pct

FROM read_parquet('{parquet_sql_path}')

GROUP BY user_id
""")

# 为用户ID建立唯一索引
con.execute("""
CREATE UNIQUE INDEX IF NOT EXISTS
idx_user_summary_user_id
ON user_summary(user_id)
""")

# 验证中间表
validation = con.execute("""
SELECT
    COUNT(*) AS user_rows,
    SUM(total_actions) AS restored_actions,
    MIN(first_action_time) AS earliest_time,
    MAX(last_action_time) AS latest_time
FROM user_summary
""").fetchdf()

preview = con.execute("""
SELECT *
FROM user_summary
ORDER BY total_actions DESC
LIMIT 10
""").fetchdf()

print("===== 用户中间表校验 =====")
display(validation)

print("\n===== 最活跃的10名用户 =====")
display(preview)


===== 用户中间表校验 =====


,user_rows,restored_actions,earliest_time,latest_time
0,10000,12256906.0,2025-11-18,2025-12-18 23:00:00



===== 最活跃的10名用户 =====


,user_id,total_actions,view_count,favorite_count,cart_count,purchase_count,unique_items,unique_categories,active_days,first_action_time,last_action_time,purchased_items,purchase_action_pct
0,36233277,31030,27720,2935,328,47,11567,372,31,2025-11-18 07:00:00,2025-12-18 23:00:00,43,0.1515
1,65645933,20770,18124,2600,27,19,7223,325,30,2025-11-18 03:00:00,2025-12-18 22:00:00,13,0.0915
2,73196588,20146,20146,0,0,0,14011,1131,31,2025-11-18 00:00:00,2025-12-18 23:00:00,0,0.0000
3,59511789,20129,18558,1274,263,34,6730,320,31,2025-11-18 10:00:00,2025-12-18 23:00:00,30,0.1689
4,130270245,19786,18857,726,185,18,5672,419,28,2025-11-18 00:00:00,2025-12-18 23:00:00,18,0.0910
5,7234861,17574,15791,1165,569,49,6149,619,31,2025-11-18 00:00:00,2025-12-18 19:00:00,48,0.2788
6,83813302,17248,16874,277,81,16,6752,311,31,2025-11-18 09:00:00,2025-12-18 22:00:00,15,0.0928
7,52577851,13419,12555,10,792,62,4757,267,31,2025-11-18 08:00:00,2025-12-18 21:00:00,51,0.4620
8,137175187,12908,12339,254,284,31,4291,217,31,2025-11-18 01:00:00,2025-12-18 23:00:00,25,0.2402
9,123842164,12705,11342,168,546,649,4053,239,31,2025-11-18 00:00:00,2025-12-18 23:00:00,299,5.1082


In [26]:
# 创建商品维度中间表
con.execute(f"""
CREATE OR REPLACE TABLE item_summary AS
SELECT
    item_id,

    MIN(item_category) AS item_category,

    COUNT(DISTINCT item_category)
        AS category_count,

    COUNT(*) AS total_actions,

    COUNT(*) FILTER (
        WHERE behavior_type = 1
    ) AS view_count,

    COUNT(*) FILTER (
        WHERE behavior_type = 2
    ) AS favorite_count,

    COUNT(*) FILTER (
        WHERE behavior_type = 3
    ) AS cart_count,

    COUNT(*) FILTER (
        WHERE behavior_type = 4
    ) AS purchase_count,

    COUNT(DISTINCT user_id)
        AS unique_users,

    COUNT(DISTINCT CASE
        WHEN behavior_type = 1
        THEN user_id
    END) AS unique_viewers,

    COUNT(DISTINCT CASE
        WHEN behavior_type = 2
        THEN user_id
    END) AS unique_favoriters,

    COUNT(DISTINCT CASE
        WHEN behavior_type = 3
        THEN user_id
    END) AS unique_cart_users,

    COUNT(DISTINCT CASE
        WHEN behavior_type = 4
        THEN user_id
    END) AS unique_buyers,

    MIN(time) AS first_action_time,

    MAX(time) AS last_action_time,

    ROUND(
        100.0
        * COUNT(*) FILTER (
            WHERE behavior_type = 4
        )
        / COUNT(*),
        4
    ) AS purchase_action_pct

FROM read_parquet('{parquet_sql_path}')

GROUP BY item_id
""")

# 为商品ID建立唯一索引
con.execute("""
CREATE UNIQUE INDEX IF NOT EXISTS
idx_item_summary_item_id
ON item_summary(item_id)
""")

# 校验商品表
item_validation = con.execute("""
SELECT
    COUNT(*) AS item_rows,
    SUM(total_actions) AS restored_actions,
    COUNT(*) FILTER (
        WHERE category_count > 1
    ) AS items_with_multiple_categories
FROM item_summary
""").fetchdf()

# 查看行为量最高的10件商品
top_items = con.execute("""
SELECT *
FROM item_summary
ORDER BY total_actions DESC
LIMIT 10
""").fetchdf()

print("===== 商品中间表校验 =====")
display(item_validation)

print("\n===== 行为量最高的10件商品 =====")
display(top_items)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

===== 商品中间表校验 =====


,item_rows,restored_actions,items_with_multiple_categories
0,2876947,12256906.0,0



===== 行为量最高的10件商品 =====


,item_id,item_category,category_count,total_actions,view_count,favorite_count,cart_count,purchase_count,unique_users,unique_viewers,unique_favoriters,unique_cart_users,unique_buyers,first_action_time,last_action_time,purchase_action_pct
0,112921337,6000,1,1445,1431,5,8,1,518,518,5,8,1,2025-11-18 00:00:00,2025-12-18 22:00:00,0.0692
1,97655171,6000,1,1289,1249,7,20,13,332,332,7,16,12,2025-11-18 00:00:00,2025-12-18 22:00:00,1.0085
2,387911330,6000,1,1102,1053,10,20,19,298,298,8,13,15,2025-11-18 05:00:00,2025-12-18 22:00:00,1.7241
3,135104537,6000,1,931,916,7,8,0,339,339,6,8,0,2025-11-18 00:00:00,2025-12-18 22:00:00,0.0000
4,14087919,5232,1,823,740,15,33,35,206,205,12,29,25,2025-11-18 09:00:00,2025-12-18 23:00:00,4.2527
5,277922302,5399,1,808,763,24,20,1,275,275,23,15,1,2025-11-18 13:00:00,2025-12-18 23:00:00,0.1238
6,2217535,6000,1,805,792,1,11,1,317,317,1,7,1,2025-11-18 00:00:00,2025-12-18 21:00:00,0.1242
7,128186279,3381,1,795,765,13,16,1,341,341,13,11,1,2025-11-18 00:00:00,2025-12-18 22:00:00,0.1258
8,5685392,3381,1,793,767,12,12,2,302,302,11,10,2,2025-11-18 01:00:00,2025-12-12 21:00:00,0.2522
9,209323160,10072,1,744,716,2,26,0,282,282,2,25,0,2025-12-09 00:00:00,2025-12-17 22:00:00,0.0000


In [27]:
# 创建小时级时间维度中间表
con.execute(f"""
CREATE OR REPLACE TABLE time_summary AS
SELECT
    time,

    CAST(time AS DATE) AS action_date,

    CAST(
        DATE_TRUNC('week', time)
        AS DATE
    ) AS week_start,

    EXTRACT(HOUR FROM time)
        AS hour_of_day,

    EXTRACT(DOW FROM time)
        AS day_of_week,

    CASE
        WHEN EXTRACT(DOW FROM time) IN (0, 6)
        THEN TRUE
        ELSE FALSE
    END AS is_weekend,

    COUNT(*) AS total_actions,

    COUNT(*) FILTER (
        WHERE behavior_type = 1
    ) AS view_count,

    COUNT(*) FILTER (
        WHERE behavior_type = 2
    ) AS favorite_count,

    COUNT(*) FILTER (
        WHERE behavior_type = 3
    ) AS cart_count,

    COUNT(*) FILTER (
        WHERE behavior_type = 4
    ) AS purchase_count,

    COUNT(DISTINCT user_id)
        AS active_users,

    COUNT(DISTINCT item_id)
        AS active_items,

    ROUND(
        100.0
        * COUNT(*) FILTER (
            WHERE behavior_type = 4
        )
        / COUNT(*),
        4
    ) AS purchase_action_pct

FROM read_parquet('{parquet_sql_path}')

GROUP BY time
""")

# 为时间建立唯一索引
con.execute("""
CREATE UNIQUE INDEX IF NOT EXISTS
idx_time_summary_time
ON time_summary(time)
""")

# 校验时间表
time_validation = con.execute("""
SELECT
    COUNT(*) AS hourly_rows,
    SUM(total_actions) AS restored_actions,
    MIN(time) AS earliest_time,
    MAX(time) AS latest_time
FROM time_summary
""").fetchdf()

# 查看行为量最高的10个小时
top_hours = con.execute("""
SELECT *
FROM time_summary
ORDER BY total_actions DESC
LIMIT 10
""").fetchdf()

print("===== 时间中间表校验 =====")
display(time_validation)

print("\n===== 行为量最高的10个小时 =====")
display(top_hours)

===== 时间中间表校验 =====


,hourly_rows,restored_actions,earliest_time,latest_time
0,744,12256906.0,2025-11-18,2025-12-18 23:00:00



===== 行为量最高的10个小时 =====


,time,action_date,week_start,hour_of_day,day_of_week,is_weekend,total_actions,view_count,favorite_count,cart_count,purchase_count,active_users,active_items,purchase_action_pct
0,2025-12-11 22:00:00,2025-12-11,2025-12-08,22,4,False,54797,51508,946,2011,332,2156,24235,0.6059
1,2025-12-12 22:00:00,2025-12-12,2025-12-08,22,5,False,51337,47748,709,1895,985,2178,22055,1.9187
2,2025-12-12 21:00:00,2025-12-12,2025-12-08,21,5,False,51259,47831,735,1843,850,2287,21861,1.6582
3,2025-12-12 00:00:00,2025-12-12,2025-12-08,0,5,False,50030,45229,584,2267,1950,1569,20261,3.8977
4,2025-12-11 23:00:00,2025-12-11,2025-12-08,23,4,False,49276,46151,958,1969,198,1706,21603,0.4018
5,2025-12-11 21:00:00,2025-12-11,2025-12-08,21,4,False,47802,45095,784,1640,283,2136,21024,0.5920
6,2025-12-12 23:00:00,2025-12-12,2025-12-08,23,5,False,46576,43091,669,1866,950,1728,20145,2.0397
7,2025-12-12 20:00:00,2025-12-12,2025-12-08,20,5,False,43206,40770,565,1228,643,2103,18506,1.4882
8,2025-12-10 21:00:00,2025-12-10,2025-12-08,21,3,False,39023,36888,782,1144,209,1718,17370,0.5356
9,2025-12-11 20:00:00,2025-12-11,2025-12-08,20,4,False,38918,36739,684,1232,263,1838,17251,0.6758


In [28]:
# 1. 日维度中间表
con.execute(f"""
CREATE OR REPLACE TABLE daily_summary AS
SELECT
    CAST(time AS DATE) AS action_date,

    EXTRACT(DOW FROM time)
        AS day_of_week,

    CASE
        WHEN EXTRACT(DOW FROM time) IN (0, 6)
        THEN TRUE
        ELSE FALSE
    END AS is_weekend,

    COUNT(*) AS total_actions,

    COUNT(*) FILTER (
        WHERE behavior_type = 1
    ) AS view_count,

    COUNT(*) FILTER (
        WHERE behavior_type = 2
    ) AS favorite_count,

    COUNT(*) FILTER (
        WHERE behavior_type = 3
    ) AS cart_count,

    COUNT(*) FILTER (
        WHERE behavior_type = 4
    ) AS purchase_count,

    COUNT(DISTINCT user_id)
        AS active_users,

    COUNT(DISTINCT item_id)
        AS active_items,

    ROUND(
        100.0
        * COUNT(*) FILTER (
            WHERE behavior_type = 4
        )
        / COUNT(*),
        4
    ) AS purchase_action_pct

FROM read_parquet('{parquet_sql_path}')

GROUP BY
    CAST(time AS DATE),
    EXTRACT(DOW FROM time)
""")

# 2. 周维度中间表
con.execute(f"""
CREATE OR REPLACE TABLE weekly_summary AS
SELECT
    CAST(
        DATE_TRUNC('week', time)
        AS DATE
    ) AS week_start,

    MIN(CAST(time AS DATE))
        AS observed_start_date,

    MAX(CAST(time AS DATE))
        AS observed_end_date,

    COUNT(DISTINCT CAST(time AS DATE))
        AS covered_days,

    COUNT(*) AS total_actions,

    COUNT(*) FILTER (
        WHERE behavior_type = 1
    ) AS view_count,

    COUNT(*) FILTER (
        WHERE behavior_type = 2
    ) AS favorite_count,

    COUNT(*) FILTER (
        WHERE behavior_type = 3
    ) AS cart_count,

    COUNT(*) FILTER (
        WHERE behavior_type = 4
    ) AS purchase_count,

    COUNT(DISTINCT user_id)
        AS active_users,

    COUNT(DISTINCT item_id)
        AS active_items,

    ROUND(
        100.0
        * COUNT(*) FILTER (
            WHERE behavior_type = 4
        )
        / COUNT(*),
        4
    ) AS purchase_action_pct

FROM read_parquet('{parquet_sql_path}')

GROUP BY
    DATE_TRUNC('week', time)
""")

# 建立索引
con.execute("""
CREATE UNIQUE INDEX IF NOT EXISTS
idx_daily_summary_date
ON daily_summary(action_date)
""")

con.execute("""
CREATE UNIQUE INDEX IF NOT EXISTS
idx_weekly_summary_week
ON weekly_summary(week_start)
""")

# 联合校验
time_grain_validation = con.execute("""
SELECT
    (SELECT COUNT(*) FROM time_summary)
        AS hourly_rows,

    (SELECT COUNT(*) FROM daily_summary)
        AS daily_rows,

    (SELECT COUNT(*) FROM weekly_summary)
        AS weekly_rows,

    (SELECT SUM(total_actions)
     FROM daily_summary)
        AS daily_restored_actions,

    (SELECT SUM(total_actions)
     FROM weekly_summary)
        AS weekly_restored_actions
""").fetchdf()

daily_preview = con.execute("""
SELECT *
FROM daily_summary
ORDER BY total_actions DESC
LIMIT 10
""").fetchdf()

weekly_preview = con.execute("""
SELECT *
FROM weekly_summary
ORDER BY week_start
""").fetchdf()

print("===== 时间粒度联合校验 =====")
display(time_grain_validation)

print("\n===== 行为量最高的10天 =====")
display(daily_preview)

print("\n===== 周度汇总 =====")
display(weekly_preview)

===== 时间粒度联合校验 =====


,hourly_rows,daily_rows,weekly_rows,daily_restored_actions,weekly_restored_actions
0,744,31,5,12256906.0,12256906.0



===== 行为量最高的10天 =====


,action_date,day_of_week,is_weekend,total_actions,view_count,favorite_count,cart_count,purchase_count,active_users,active_items,purchase_action_pct
0,2025-12-12,5,False,691712,641507,10446,24508,15251,7720,232398,2.2048
1,2025-12-11,4,False,488508,460329,9310,15643,3226,6894,180622,0.6604
2,2025-12-10,3,False,421910,397661,8753,12280,3216,6652,162006,0.7622
3,2025-12-03,3,False,411606,387497,8492,11732,3885,6585,156706,0.9439
4,2025-12-13,6,True,407160,385337,7859,10486,3478,6776,156418,0.8542
5,2025-12-02,2,False,405216,382052,8022,11521,3621,6550,154480,0.8936
6,2025-12-14,0,True,402541,380717,8128,10213,3483,6668,155900,0.8653
7,2025-11-30,0,True,401620,379439,7786,10779,3616,6379,155985,0.9004
8,2025-12-04,4,False,399952,376307,8559,11395,3691,6531,153369,0.9229
9,2025-12-07,0,True,399751,376596,8632,11267,3256,6422,152758,0.8145



===== 周度汇总 =====


,week_start,observed_start_date,observed_end_date,covered_days,total_actions,view_count,favorite_count,cart_count,purchase_count,active_users,active_items,purchase_action_pct
0,2025-11-17,2025-11-18,2025-11-23,6,2156114,2032873,43009,59416,20816,8964,681413,0.9654
1,2025-11-24,2025-11-24,2025-11-30,7,2587816,2442624,51332,69682,24178,9224,801899,0.9343
2,2025-12-01,2025-12-01,2025-12-07,7,2762624,2602649,56821,78201,24953,9354,832895,0.9032
3,2025-12-08,2025-12-08,2025-12-14,7,3196523,3003909,61284,95808,35522,9506,920810,1.1113
4,2025-12-15,2025-12-15,2025-12-18,4,1553829,1468526,30110,40457,14736,8947,514725,0.9484


In [29]:
from pathlib import Path

processed_dir = Path("../data/processed").resolve()
reports_dir = Path("../reports").resolve()

processed_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)

# 需要导出的中间表
export_jobs = {
    "user_summary": "user_summary_v1.parquet",
    "item_summary": "item_summary_v1.parquet",
    "time_summary": "time_hourly_summary_v1.parquet",
    "daily_summary": "time_daily_summary_v1.parquet",
    "weekly_summary": "time_weekly_summary_v1.parquet"
}

print("===== 导出中间表 =====")

for table_name, file_name in export_jobs.items():
    output_path = processed_dir / file_name

    if output_path.exists():
        print(f"已存在，跳过：{file_name}")
    else:
        sql_output_path = output_path.as_posix()

        con.execute(f"""
        COPY {table_name}
        TO '{sql_output_path}'
        (FORMAT PARQUET, COMPRESSION SNAPPY)
        """)

        print(f"导出成功：{file_name}")

# 获取各表行数
table_counts = {}

for table_name in export_jobs:
    row_count = con.execute(
        f"SELECT COUNT(*) FROM {table_name}"
    ).fetchone()[0]

    table_counts[table_name] = row_count

# 生成中间表设计说明
DESIGN_REPORT_PATH = (
    reports_dir
    / "intermediate_table_design.md"
)

design_report = f"""# 多维度中间表设计说明

## 1. 设计目标

基于标准化用户行为数据构建面向分析的聚合中间表，
降低后续探索性分析、特征工程和模型训练的重复计算成本。

原始行为明细不会删除，仍保存在标准化基础数据集中。
重复浏览、收藏、加购和购买行为在中间表中体现为行为次数。

## 2. 数据模型

标准化行为明细表是核心事实表，包含以下关联字段：

- `user_id`
- `item_id`
- `item_category`
- `time`
- `behavior_type`

不同中间表使用与自身统计粒度相匹配的主键，
不使用统一的复合主键。

## 3. 中间表说明

| 表名 | 统计粒度 | 主键 | 行数 | 主要指标 |
|---|---|---|---:|---|
| user_summary | 每名用户一行 | user_id | {table_counts["user_summary"]:,} | 浏览、收藏、加购、购买、活跃天数、商品覆盖 |
| item_summary | 每件商品一行 | item_id | {table_counts["item_summary"]:,} | 流量、互动、购买、独立用户和买家数 |
| time_summary | 每小时一行 | time | {table_counts["time_summary"]:,} | 小时行为量、活跃用户和活跃商品 |
| daily_summary | 每天一行 | action_date | {table_counts["daily_summary"]:,} | 日行为趋势和购买行为占比 |
| weekly_summary | 每周一行 | week_start | {table_counts["weekly_summary"]:,} | 周度趋势和覆盖天数 |

## 4. 关联方式

- 行为明细表通过 `user_id` 关联用户中间表；
- 行为明细表通过 `item_id` 关联商品中间表；
- 行为明细表通过 `time` 关联小时中间表；
- 通过日期转换关联日表和周表；
- 商品的品类字段为 `item_category`。

## 5. 重复行为处理

没有直接删除重复行为。

同一用户对同一商品的重复行为被汇总为：

- `view_count`
- `favorite_count`
- `cart_count`
- `purchase_count`

这些指标反映兴趣强度、购买意愿和复购特征。

## 6. 指标口径说明

`purchase_action_pct` 的计算方式为：

购买行为次数 ÷ 全部行为次数 × 100%

该指标表示购买行为在全部行为中的占比，
不直接等同于严格意义上的用户购买转化率。

## 7. 数据一致性校验

| 校验项目 | 结果 |
|---|---:|
| 用户表行数 | {table_counts["user_summary"]:,} |
| 商品表行数 | {table_counts["item_summary"]:,} |
| 小时表行数 | {table_counts["time_summary"]:,} |
| 日表行数 | {table_counts["daily_summary"]:,} |
| 周表行数 | {table_counts["weekly_summary"]:,} |
| 用户表还原行为数 | 12,256,906 |
| 商品表还原行为数 | 12,256,906 |
| 日表还原行为数 | 12,256,906 |
| 周表还原行为数 | 12,256,906 |
| 多品类归属商品数 | 0 |

## 8. 时间分析注意事项

数据覆盖2025年11月18日至12月18日。

首个自然周只覆盖6天，最后一个自然周只覆盖4天，
因此进行周度比较时应使用日均行为量，不能只比较周度总量。

12月12日的行为量和购买行为占比明显升高，
与“双12”时间窗口重合，后续需结合活动背景进一步验证，
不能仅凭时间重合直接认定因果关系。
"""

DESIGN_REPORT_PATH.write_text(
    design_report,
    encoding="utf-8"
)

print("\n===== 步骤二产出物 =====")

for file_name in export_jobs.values():
    path = processed_dir / file_name
    size_mb = path.stat().st_size / 1024**2

    print(f"{file_name}: {size_mb:.2f} MB")

print(
    "设计说明：",
    DESIGN_REPORT_PATH
)

===== 导出中间表 =====
导出成功：user_summary_v1.parquet
导出成功：item_summary_v1.parquet
导出成功：time_hourly_summary_v1.parquet
导出成功：time_daily_summary_v1.parquet
导出成功：time_weekly_summary_v1.parquet

===== 步骤二产出物 =====
user_summary_v1.parquet: 0.27 MB
item_summary_v1.parquet: 31.72 MB
time_hourly_summary_v1.parquet: 0.03 MB
time_daily_summary_v1.parquet: 0.00 MB
time_weekly_summary_v1.parquet: 0.00 MB
设计说明： /Users/nyx/Desktop/user-behavior-analysis/reports/intermediate_table_design.md


In [30]:
from pathlib import Path

# 构建品类维度中间表
con.execute(f"""
CREATE OR REPLACE TABLE category_summary AS
SELECT
    item_category,

    COUNT(*) AS total_actions,

    COUNT(*) FILTER (
        WHERE behavior_type = 1
    ) AS view_count,

    COUNT(*) FILTER (
        WHERE behavior_type = 2
    ) AS favorite_count,

    COUNT(*) FILTER (
        WHERE behavior_type = 3
    ) AS cart_count,

    COUNT(*) FILTER (
        WHERE behavior_type = 4
    ) AS purchase_count,

    COUNT(DISTINCT user_id)
        AS unique_users,

    COUNT(DISTINCT item_id)
        AS unique_items,

    COUNT(DISTINCT CASE
        WHEN behavior_type = 4
        THEN user_id
    END) AS unique_buyers,

    COUNT(DISTINCT CASE
        WHEN behavior_type = 4
        THEN item_id
    END) AS purchased_items,

    MIN(time) AS first_action_time,

    MAX(time) AS last_action_time,

    ROUND(
        100.0
        * COUNT(*) FILTER (
            WHERE behavior_type = 4
        )
        / COUNT(*),
        4
    ) AS purchase_action_pct

FROM read_parquet('{parquet_sql_path}')

GROUP BY item_category
""")

# 建立唯一索引
con.execute("""
CREATE UNIQUE INDEX IF NOT EXISTS
idx_category_summary_category
ON category_summary(item_category)
""")

# 数据一致性校验
category_validation = con.execute("""
SELECT
    COUNT(*) AS category_rows,
    SUM(total_actions) AS restored_actions,
    SUM(unique_items) AS restored_items,
    MIN(first_action_time) AS earliest_time,
    MAX(last_action_time) AS latest_time
FROM category_summary
""").fetchdf()

# 查看行为量最高的10个品类
top_categories = con.execute("""
SELECT *
FROM category_summary
ORDER BY total_actions DESC
LIMIT 10
""").fetchdf()

print("===== 品类中间表校验 =====")
display(category_validation)

print("\n===== 行为量最高的10个品类 =====")
display(top_categories)

# 导出完整品类表
category_output = Path(
    "../data/processed/category_summary_v1.parquet"
)

if category_output.exists():
    print("\n文件已经存在：", category_output)
else:
    con.execute("""
    COPY category_summary
    TO '../data/processed/category_summary_v1.parquet'
    (FORMAT PARQUET, COMPRESSION SNAPPY)
    """)

    print("\n品类表已导出：", category_output)

===== 品类中间表校验 =====


,category_rows,restored_actions,restored_items,earliest_time,latest_time
0,8916,12256906.0,2876947.0,2025-11-18,2025-12-18 23:00:00



===== 行为量最高的10个品类 =====


,item_category,total_actions,view_count,favorite_count,cart_count,purchase_count,unique_users,unique_items,unique_buyers,purchased_items,first_action_time,last_action_time,purchase_action_pct
0,1863,393247,371738,10200,9309,2000,5997,83354,1252,1674,2025-11-18,2025-12-18 23:00:00,0.5086
1,13230,356773,342694,7226,6012,841,5619,74067,564,763,2025-11-18,2025-12-18 23:00:00,0.2357
2,5027,334280,320870,6988,5564,858,5499,60323,627,739,2025-11-18,2025-12-18 23:00:00,0.2567
3,5894,330207,314784,7850,6615,958,5279,83758,605,884,2025-11-18,2025-12-18 23:00:00,0.2901
4,6513,295768,281370,6688,6651,1059,5288,65045,698,915,2025-11-18,2025-12-18 23:00:00,0.3581
5,5399,281739,268639,6616,5430,1054,5312,54559,651,896,2025-11-18,2025-12-18 23:00:00,0.3741
6,11279,187056,177961,4687,3686,722,4715,47538,585,649,2025-11-18,2025-12-18 23:00:00,0.3860
7,2825,163626,155949,3360,3692,625,4551,43997,413,538,2025-11-18,2025-12-18 23:00:00,0.3820
8,5232,144200,135506,2597,4486,1611,4972,25900,954,1004,2025-11-18,2025-12-18 23:00:00,1.1172
9,10894,138782,131221,2802,3894,865,4523,33482,592,717,2025-11-18,2025-12-18 23:00:00,0.6233



品类表已导出： ../data/processed/category_summary_v1.parquet


In [31]:
from pathlib import Path

CUTOFF_TIME = "2025-12-18 00:00:00"
LABEL_END = "2025-12-19 00:00:00"

print("观测窗口结束：", CUTOFF_TIME)
print("标签窗口结束：", LABEL_END)

con.execute(f"""
CREATE OR REPLACE TABLE user_item_summary AS

WITH observation_data AS (
    SELECT *
    FROM read_parquet('{parquet_sql_path}')
    WHERE time < TIMESTAMP '{CUTOFF_TIME}'
),

base_features AS (
    SELECT
        user_id,
        item_id,

        MIN(item_category)
            AS item_category,

        COUNT(*)
            AS total_actions,

        COUNT(*) FILTER (
            WHERE behavior_type = 1
        ) AS view_count,

        COUNT(*) FILTER (
            WHERE behavior_type = 2
        ) AS favorite_count,

        COUNT(*) FILTER (
            WHERE behavior_type = 3
        ) AS cart_count,

        COUNT(*) FILTER (
            WHERE behavior_type = 4
        ) AS purchase_count,

        MIN(time)
            AS first_interaction_time,

        MAX(time)
            AS last_interaction_time,

        COUNT(
            DISTINCT CAST(time AS DATE)
        ) AS interaction_days,

        COUNT(DISTINCT time)
            AS active_hours

    FROM observation_data

    GROUP BY
        user_id,
        item_id
),

purchase_labels AS (
    SELECT
        user_id,
        item_id,
        CAST(1 AS UTINYINT)
            AS purchase_next_horizon_label

    FROM read_parquet('{parquet_sql_path}')

    WHERE
        time >= TIMESTAMP '{CUTOFF_TIME}'
        AND time < TIMESTAMP '{LABEL_END}'
        AND behavior_type = 4

    GROUP BY
        user_id,
        item_id
)

SELECT
    base.user_id,
    base.item_id,
    base.item_category,
    base.total_actions,
    base.view_count,
    base.favorite_count,
    base.cart_count,
    base.purchase_count,
    base.first_interaction_time,
    base.last_interaction_time,
    base.interaction_days,
    base.active_hours,

    GREATEST(
        base.view_count - 1,
        0
    ) AS repeat_view_count,

    DATE_DIFF(
        'hour',
        base.last_interaction_time,
        TIMESTAMP '{CUTOFF_TIME}'
    ) AS recency_hours,

    COALESCE(
        labels.purchase_next_horizon_label,
        0
    ) AS purchase_next_horizon_label

FROM base_features AS base

LEFT JOIN purchase_labels AS labels
    ON base.user_id = labels.user_id
    AND base.item_id = labels.item_id
""")

观测窗口结束： 2025-12-18 00:00:00
标签窗口结束： 2025-12-19 00:00:00


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [32]:
# 用户—商品表基础校验
user_item_validation = con.execute(f"""
SELECT
    COUNT(*) AS user_item_rows,

    SUM(total_actions)
        AS restored_observation_actions,

    (
        SELECT COUNT(*)
        FROM read_parquet('{parquet_sql_path}')
        WHERE time < TIMESTAMP '{CUTOFF_TIME}'
    ) AS expected_observation_actions,

    SUM(purchase_next_horizon_label)
        AS positive_labels,

    ROUND(
        100.0
        * SUM(purchase_next_horizon_label)
        / COUNT(*),
        6
    ) AS positive_rate_pct,

    MIN(recency_hours)
        AS min_recency_hours,

    MAX(recency_hours)
        AS max_recency_hours

FROM user_item_summary
""").fetchdf()

# 检查预测日购买商品是否在候选集中
candidate_coverage = con.execute(f"""
WITH prediction_day_purchases AS (
    SELECT DISTINCT
        user_id,
        item_id

    FROM read_parquet('{parquet_sql_path}')

    WHERE
        time >= TIMESTAMP '{CUTOFF_TIME}'
        AND time < TIMESTAMP '{LABEL_END}'
        AND behavior_type = 4
)

SELECT
    COUNT(*) AS prediction_purchase_pairs,

    COUNT(summary.user_id)
        AS covered_purchase_pairs,

    ROUND(
        100.0
        * COUNT(summary.user_id)
        / COUNT(*),
        4
    ) AS candidate_coverage_pct

FROM prediction_day_purchases AS purchases

LEFT JOIN user_item_summary AS summary
    ON purchases.user_id = summary.user_id
    AND purchases.item_id = summary.item_id
""").fetchdf()

positive_examples = con.execute("""
SELECT *
FROM user_item_summary
WHERE purchase_next_horizon_label = 1
ORDER BY total_actions DESC
LIMIT 10
""").fetchdf()

print("===== 用户—商品表校验 =====")
display(user_item_validation)

print("\n===== 预测日购买候选覆盖率 =====")
display(candidate_coverage)

print("\n===== 正样本示例 =====")
display(positive_examples)

===== 用户—商品表校验 =====


,user_item_rows,restored_observation_actions,expected_observation_actions,positive_labels,positive_rate_pct,min_recency_hours,max_recency_hours
0,4546934,11881309.0,11881309,1028.0,0.022609,1,720



===== 预测日购买候选覆盖率 =====


,prediction_purchase_pairs,covered_purchase_pairs,candidate_coverage_pct
0,3151,1028,32.6246



===== 正样本示例 =====


,user_id,item_id,item_category,total_actions,view_count,favorite_count,cart_count,purchase_count,first_interaction_time,last_interaction_time,interaction_days,active_hours,repeat_view_count,recency_hours,purchase_next_horizon_label
0,71883073,58558840,1083,57,45,0,0,12,2025-11-19 12:00:00,2025-12-14 19:00:00,14,16,44,77,1
1,137998405,143865903,6307,57,53,0,3,1,2025-12-05 21:00:00,2025-12-17 09:00:00,8,12,52,15,1
2,131573117,8788846,10223,56,50,0,5,1,2025-11-24 10:00:00,2025-12-15 19:00:00,14,21,49,53,1
3,83570876,255708467,5027,56,53,2,1,0,2025-12-05 15:00:00,2025-12-17 15:00:00,13,26,52,9,1
4,71883073,380344970,1083,47,32,0,0,15,2025-11-19 12:00:00,2025-12-14 16:00:00,8,15,31,80,1
5,62073731,25237777,9396,44,42,1,1,0,2025-11-20 11:00:00,2025-12-17 23:00:00,13,17,41,1,1
6,70212546,249750060,1083,41,30,0,4,7,2025-11-25 10:00:00,2025-12-17 15:00:00,8,11,29,9,1
7,37780288,316255438,10961,35,34,0,1,0,2025-11-30 16:00:00,2025-12-17 19:00:00,7,8,33,5,1
8,108260948,71382075,7767,35,32,0,0,3,2025-11-23 16:00:00,2025-12-15 16:00:00,6,8,31,56,1
9,78462766,198971992,11135,34,31,0,0,3,2025-11-27 19:00:00,2025-12-16 17:00:00,10,11,30,31,1


In [33]:
candidate_diagnosis = con.execute(f"""
WITH prediction_purchases AS (
    SELECT DISTINCT
        user_id,
        item_id,
        item_category

    FROM read_parquet('{parquet_sql_path}')

    WHERE
        time >= TIMESTAMP '{CUTOFF_TIME}'
        AND time < TIMESTAMP '{LABEL_END}'
        AND behavior_type = 4
),

historical_pairs AS (
    SELECT DISTINCT
        user_id,
        item_id

    FROM read_parquet('{parquet_sql_path}')

    WHERE time < TIMESTAMP '{CUTOFF_TIME}'
),

historical_items AS (
    SELECT DISTINCT
        item_id

    FROM read_parquet('{parquet_sql_path}')

    WHERE time < TIMESTAMP '{CUTOFF_TIME}'
),

historical_user_categories AS (
    SELECT DISTINCT
        user_id,
        item_category

    FROM read_parquet('{parquet_sql_path}')

    WHERE time < TIMESTAMP '{CUTOFF_TIME}'
),

recent_pairs AS (
    SELECT DISTINCT
        user_id,
        item_id

    FROM read_parquet('{parquet_sql_path}')

    WHERE
        time >= TIMESTAMP '{CUTOFF_TIME}'
               - INTERVAL 7 DAY
        AND time < TIMESTAMP '{CUTOFF_TIME}'
),

recent_user_categories AS (
    SELECT DISTINCT
        user_id,
        item_category

    FROM read_parquet('{parquet_sql_path}')

    WHERE
        time >= TIMESTAMP '{CUTOFF_TIME}'
               - INTERVAL 7 DAY
        AND time < TIMESTAMP '{CUTOFF_TIME}'
),

diagnosis AS (
    SELECT
        purchases.*,

        CASE WHEN historical_pairs.user_id
                  IS NOT NULL
             THEN 1 ELSE 0
        END AS pair_seen_before,

        CASE WHEN historical_items.item_id
                  IS NOT NULL
             THEN 1 ELSE 0
        END AS item_seen_before,

        CASE WHEN historical_user_categories.user_id
                  IS NOT NULL
             THEN 1 ELSE 0
        END AS category_seen_by_user,

        CASE WHEN recent_pairs.user_id
                  IS NOT NULL
             THEN 1 ELSE 0
        END AS pair_seen_last_7d,

        CASE WHEN recent_user_categories.user_id
                  IS NOT NULL
             THEN 1 ELSE 0
        END AS category_seen_last_7d

    FROM prediction_purchases AS purchases

    LEFT JOIN historical_pairs
        ON purchases.user_id
           = historical_pairs.user_id
        AND purchases.item_id
           = historical_pairs.item_id

    LEFT JOIN historical_items
        ON purchases.item_id
           = historical_items.item_id

    LEFT JOIN historical_user_categories
        ON purchases.user_id
           = historical_user_categories.user_id
        AND purchases.item_category
           = historical_user_categories.item_category

    LEFT JOIN recent_pairs
        ON purchases.user_id
           = recent_pairs.user_id
        AND purchases.item_id
           = recent_pairs.item_id

    LEFT JOIN recent_user_categories
        ON purchases.user_id
           = recent_user_categories.user_id
        AND purchases.item_category
           = recent_user_categories.item_category
)

SELECT
    COUNT(*) AS total_purchase_pairs,

    SUM(pair_seen_before)
        AS pair_seen_before,

    ROUND(
        100.0 * SUM(pair_seen_before) / COUNT(*),
        4
    ) AS pair_seen_pct,

    SUM(item_seen_before)
        AS item_seen_before,

    ROUND(
        100.0 * SUM(item_seen_before) / COUNT(*),
        4
    ) AS item_seen_pct,

    SUM(category_seen_by_user)
        AS category_seen_by_user,

    ROUND(
        100.0
        * SUM(category_seen_by_user)
        / COUNT(*),
        4
    ) AS category_seen_pct,

    SUM(pair_seen_last_7d)
        AS pair_seen_last_7d,

    SUM(category_seen_last_7d)
        AS category_seen_last_7d

FROM diagnosis
""").fetchdf()

display(candidate_diagnosis)

,total_purchase_pairs,pair_seen_before,pair_seen_pct,item_seen_before,item_seen_pct,category_seen_by_user,category_seen_pct,pair_seen_last_7d,category_seen_last_7d
0,3151,1028.0,32.6246,2022.0,64.1701,2116.0,67.1533,861.0,1700.0


In [34]:
candidate_strategy_comparison = con.execute(f"""
WITH prediction_purchases AS (
    SELECT DISTINCT
        user_id,
        item_id,
        item_category

    FROM read_parquet('{parquet_sql_path}')

    WHERE
        time >= TIMESTAMP '{CUTOFF_TIME}'
        AND time < TIMESTAMP '{LABEL_END}'
        AND behavior_type = 4
),

observation_data AS (
    SELECT *
    FROM read_parquet('{parquet_sql_path}')
    WHERE time < TIMESTAMP '{CUTOFF_TIME}'
),

historical_pairs AS (
    SELECT DISTINCT
        user_id,
        item_id

    FROM observation_data
),

historical_user_categories AS (
    SELECT DISTINCT
        user_id,
        item_category

    FROM observation_data
),

recent_category_stats AS (
    SELECT
        user_id,
        item_category,
        COUNT(*) AS category_actions

    FROM observation_data

    WHERE
        time >= TIMESTAMP '{CUTOFF_TIME}'
               - INTERVAL 7 DAY

    GROUP BY
        user_id,
        item_category
),

recent_category_rank AS (
    SELECT
        user_id,
        item_category,

        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY
                category_actions DESC,
                item_category
        ) AS user_category_rank

    FROM recent_category_stats
),

item_stats AS (
    SELECT
        item_id,
        MIN(item_category) AS item_category,

        COUNT(*) AS total_actions,

        COUNT(*) FILTER (
            WHERE behavior_type = 3
        ) AS cart_count,

        COUNT(*) FILTER (
            WHERE behavior_type = 4
        ) AS purchase_count

    FROM observation_data

    GROUP BY item_id
),

item_rank AS (
    SELECT
        item_id,
        item_category,

        ROW_NUMBER() OVER (
            PARTITION BY item_category
            ORDER BY
                purchase_count DESC,
                cart_count DESC,
                total_actions DESC,
                item_id
        ) AS category_item_rank,

        ROW_NUMBER() OVER (
            ORDER BY
                purchase_count DESC,
                cart_count DESC,
                total_actions DESC,
                item_id
        ) AS global_item_rank

    FROM item_stats
),

purchase_flags AS (
    SELECT
        purchases.user_id,
        purchases.item_id,
        purchases.item_category,

        CASE
            WHEN historical_pairs.user_id
                 IS NOT NULL
            THEN 1 ELSE 0
        END AS historical_pair,

        CASE
            WHEN historical_user_categories.user_id
                 IS NOT NULL
            THEN 1 ELSE 0
        END AS historical_category,

        recent_category_rank.user_category_rank,

        item_rank.category_item_rank,
        item_rank.global_item_rank

    FROM prediction_purchases AS purchases

    LEFT JOIN historical_pairs
        ON purchases.user_id
           = historical_pairs.user_id
        AND purchases.item_id
           = historical_pairs.item_id

    LEFT JOIN historical_user_categories
        ON purchases.user_id
           = historical_user_categories.user_id
        AND purchases.item_category
           = historical_user_categories.item_category

    LEFT JOIN recent_category_rank
        ON purchases.user_id
           = recent_category_rank.user_id
        AND purchases.item_category
           = recent_category_rank.item_category

    LEFT JOIN item_rank
        ON purchases.item_id
           = item_rank.item_id
),

strategy_results AS (
    SELECT
        '仅历史用户—商品组合'
            AS strategy,

        COUNT(*) FILTER (
            WHERE historical_pair = 1
        ) AS covered_pairs

    FROM purchase_flags

    UNION ALL

    SELECT
        '历史组合＋近期Top5品类×品类Top20商品',

        COUNT(*) FILTER (
            WHERE
                historical_pair = 1
                OR (
                    user_category_rank <= 5
                    AND category_item_rank <= 20
                )
        )

    FROM purchase_flags

    UNION ALL

    SELECT
        '历史组合＋近期Top5品类×品类Top50商品',

        COUNT(*) FILTER (
            WHERE
                historical_pair = 1
                OR (
                    user_category_rank <= 5
                    AND category_item_rank <= 50
                )
        )

    FROM purchase_flags

    UNION ALL

    SELECT
        '历史组合＋近期Top10品类×品类Top50商品',

        COUNT(*) FILTER (
            WHERE
                historical_pair = 1
                OR (
                    user_category_rank <= 10
                    AND category_item_rank <= 50
                )
        )

    FROM purchase_flags

    UNION ALL

    SELECT
        '历史组合＋近期Top10品类×品类Top100商品',

        COUNT(*) FILTER (
            WHERE
                historical_pair = 1
                OR (
                    user_category_rank <= 10
                    AND category_item_rank <= 100
                )
        )

    FROM purchase_flags

    UNION ALL

    SELECT
        '历史组合＋近期Top10×Top100＋全局Top500',

        COUNT(*) FILTER (
            WHERE
                historical_pair = 1
                OR (
                    user_category_rank <= 10
                    AND category_item_rank <= 100
                )
                OR global_item_rank <= 500
        )

    FROM purchase_flags

    UNION ALL

    SELECT
        '理论上限：历史出现商品＋用户历史接触品类',

        COUNT(*) FILTER (
            WHERE
                historical_pair = 1
                OR (
                    historical_category = 1
                    AND category_item_rank IS NOT NULL
                )
        )

    FROM purchase_flags
)

SELECT
    strategy,
    covered_pairs,

    3151 - covered_pairs
        AS uncovered_pairs,

    ROUND(
        100.0 * covered_pairs / 3151,
        4
    ) AS coverage_pct

FROM strategy_results

ORDER BY coverage_pct
""").fetchdf()

display(candidate_strategy_comparison)

,strategy,covered_pairs,uncovered_pairs,coverage_pct
0,仅历史用户—商品组合,1028,2123,32.6246
1,历史组合＋近期Top5品类×品类Top20商品,1037,2114,32.9102
2,历史组合＋近期Top5品类×品类Top50商品,1045,2106,33.1641
3,历史组合＋近期Top10品类×品类Top50商品,1053,2098,33.4180
4,历史组合＋近期Top10品类×品类Top100商品,1069,2082,33.9257
5,历史组合＋近期Top10×Top100＋全局Top500,1096,2055,34.7826
6,理论上限：历史出现商品＋用户历史接触品类,1538,1613,48.8099


In [35]:
from pathlib import Path

# 只保留观测窗口特征，不包含预测标签
con.execute("""
CREATE OR REPLACE TABLE user_item_features AS
SELECT
    user_id,
    item_id,
    item_category,
    total_actions,
    view_count,
    favorite_count,
    cart_count,
    purchase_count,
    first_interaction_time,
    last_interaction_time,
    interaction_days,
    active_hours,
    repeat_view_count,
    recency_hours
FROM user_item_summary
""")

# 单独保存预测标签
con.execute("""
CREATE OR REPLACE TABLE user_item_labels AS
SELECT
    user_id,
    item_id,
    purchase_next_horizon_label
FROM user_item_summary
""")

# 校验特征表与标签表
step2_validation = con.execute("""
SELECT
    (SELECT COUNT(*)
     FROM user_item_features)
        AS feature_rows,

    (SELECT COUNT(*)
     FROM user_item_labels)
        AS label_rows,

    (SELECT SUM(total_actions)
     FROM user_item_features)
        AS restored_actions,

    (SELECT SUM(
        purchase_next_horizon_label
     )
     FROM user_item_labels)
        AS positive_labels
""").fetchdf()

display(step2_validation)

# 导出纯特征表
feature_output = Path(
    "../data/processed/user_item_features_v1.parquet"
)

if feature_output.exists():
    print("特征文件已经存在：", feature_output)
else:
    con.execute("""
    COPY user_item_features
    TO '../data/processed/user_item_features_v1.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD)
    """)

    print("特征表已导出：", feature_output)

# 标签单独保存，供步骤三使用
label_output = Path(
    "../data/processed/user_item_labels_v1.parquet"
)

if label_output.exists():
    print("标签文件已经存在：", label_output)
else:
    con.execute("""
    COPY user_item_labels
    TO '../data/processed/user_item_labels_v1.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD)
    """)

    print("标签表已导出：", label_output)

print(
    f"特征文件大小："
    f"{feature_output.stat().st_size / 1024**2:.2f} MB"
)

print(
    f"标签文件大小："
    f"{label_output.stat().st_size / 1024**2:.2f} MB"
)

,feature_rows,label_rows,restored_actions,positive_labels
0,4546934,4546934,11881309.0,1028.0


特征表已导出： ../data/processed/user_item_features_v1.parquet
标签表已导出： ../data/processed/user_item_labels_v1.parquet
特征文件大小：57.17 MB
标签文件大小：26.28 MB


In [36]:
from pathlib import Path
import pandas as pd

reports_dir = Path("../reports")
reports_dir.mkdir(parents=True, exist_ok=True)

# 获取各中间表实际行数
step2_table_summary = pd.DataFrame([
    {
        "中间表": "user_summary",
        "统计粒度": "每名用户一行",
        "主键": "user_id",
        "行数": con.execute(
            "SELECT COUNT(*) FROM user_summary"
        ).fetchone()[0],
        "状态": "已完成"
    },
    {
        "中间表": "item_summary",
        "统计粒度": "每件商品一行",
        "主键": "item_id",
        "行数": con.execute(
            "SELECT COUNT(*) FROM item_summary"
        ).fetchone()[0],
        "状态": "已完成"
    },
    {
        "中间表": "category_summary",
        "统计粒度": "每个品类一行",
        "主键": "item_category",
        "行数": con.execute(
            "SELECT COUNT(*) FROM category_summary"
        ).fetchone()[0],
        "状态": "已完成"
    },
    {
        "中间表": "time_hourly_summary",
        "统计粒度": "每小时一行",
        "主键": "time",
        "行数": con.execute(
            "SELECT COUNT(*) FROM time_summary"
        ).fetchone()[0],
        "状态": "已完成"
    },
    {
        "中间表": "time_daily_summary",
        "统计粒度": "每天一行",
        "主键": "action_date",
        "行数": con.execute(
            "SELECT COUNT(*) FROM daily_summary"
        ).fetchone()[0],
        "状态": "已完成"
    },
    {
        "中间表": "time_weekly_summary",
        "统计粒度": "每周一行",
        "主键": "week_start",
        "行数": con.execute(
            "SELECT COUNT(*) FROM weekly_summary"
        ).fetchone()[0],
        "状态": "已完成"
    },
    {
        "中间表": "user_item_features",
        "统计粒度": "每个历史用户—商品组合一行",
        "主键": "user_id + item_id",
        "行数": con.execute(
            "SELECT COUNT(*) FROM user_item_features"
        ).fetchone()[0],
        "状态": "已完成"
    },
    {
        "中间表": "user_item_labels",
        "统计粒度": "每个历史用户—商品组合一行",
        "主键": "user_id + item_id",
        "行数": con.execute(
            "SELECT COUNT(*) FROM user_item_labels"
        ).fetchone()[0],
        "状态": "已完成，步骤三使用"
    }
])

display(step2_table_summary)

# 保存校验汇总表
summary_csv_path = (
    reports_dir
    / "step2_intermediate_table_summary.csv"
)

step2_table_summary.to_csv(
    summary_csv_path,
    index=False,
    encoding="utf-8-sig"
)

# 生成正式步骤二报告
report_path = (
    reports_dir
    / "intermediate_table_design.md"
)

report_text = f"""# 多维度中间表构建与设计报告

## 1. 步骤目标

基于步骤一生成的标准化Parquet基础数据，
构建用户、商品、品类、时间和用户—商品维度中间表，
降低后续分析和特征计算的重复成本。

本步骤不训练模型。

## 2. 基础数据

| 指标 | 结果 |
|---|---:|
| 原始行为记录数 | 12,256,906 |
| 用户数 | 10,000 |
| 商品数 | 2,876,947 |
| 品类数 | 8,916 |
| 开始时间 | 2025-11-18 00:00:00 |
| 结束时间 | 2025-12-18 23:00:00 |

## 3. 中间表结构

| 中间表 | 统计粒度 | 主键 | 行数 | 状态 |
|---|---|---|---:|---|
| user_summary | 每名用户一行 | user_id | 10,000 | 已完成 |
| item_summary | 每件商品一行 | item_id | 2,876,947 | 已完成 |
| category_summary | 每个品类一行 | item_category | 8,916 | 已完成 |
| time_hourly_summary | 每小时一行 | time | 744 | 已完成 |
| time_daily_summary | 每天一行 | action_date | 31 | 已完成 |
| time_weekly_summary | 每周一行 | week_start | 5 | 已完成 |
| user_item_features | 每个历史用户—商品组合一行 | user_id + item_id | 4,546,934 | 已完成 |
| user_item_labels | 每个历史用户—商品组合一行 | user_id + item_id | 4,546,934 | 已完成，步骤三使用 |

## 4. 行为统计口径

行为编码定义：

- 1：浏览；
- 2：收藏；
- 3：加购；
- 4：购买。

重复行为没有直接删除，而是分别统计为：

- view_count；
- favorite_count；
- cart_count；
- purchase_count；
- repeat_view_count。

重复行为用于衡量用户兴趣强度和购买意向。

## 5. 各表主要指标

### 5.1 用户表

包括总交互数、四类行为次数、不同商品数、
不同品类数、活跃天数、首次和末次行为时间、
购买商品数及购买行为占比。

### 5.2 商品表

包括商品总流量、四类行为次数、独立交互用户数、
独立浏览用户数、独立收藏用户数、独立加购用户数、
独立购买用户数及购买行为占比。

### 5.3 品类表

包括品类总交互数、四类行为次数、商品数、
独立用户数、独立购买用户数和产生购买的商品数。

### 5.4 时间表

按照小时、日和周三个层级统计平台行为趋势，
包括总行为量、四类行为次数、活跃用户数和活跃商品数。

首周仅覆盖6天，末周仅覆盖4天，
因此周度比较不能只比较总量。

### 5.5 用户—商品表

观测窗口为2025年11月18日至12月17日，
以2025年12月18日作为预测窗口。

特征表与标签表已经物理分离：

- user_item_features只包含预测时点之前的行为特征；
- user_item_labels只保存预测窗口购买标签；
- 标签不得作为模型输入特征。

## 6. 一致性校验

| 校验项目 | 结果 |
|---|---:|
| 用户表还原行为数 | 12,256,906 |
| 商品表还原行为数 | 12,256,906 |
| 品类表还原行为数 | 12,256,906 |
| 日表还原行为数 | 12,256,906 |
| 周表还原行为数 | 12,256,906 |
| 观测窗口行为数 | 11,881,309 |
| 用户—商品特征表还原行为数 | 11,881,309 |
| 用户—商品正标签数 | 1,028 |
| 商品对应多个品类的异常数量 | 0 |

所有中间表均通过行数、行为总量、时间范围和主键粒度校验。

## 7. 输出文件

标准化和中间表文件保存在data/processed目录：

- user_behavior_standardized_v1.parquet
- user_summary_v1.parquet
- item_summary_v1.parquet
- category_summary_v1.parquet
- time_hourly_summary_v1.parquet
- time_daily_summary_v1.parquet
- time_weekly_summary_v1.parquet
- user_item_features_v1.parquet
- user_item_labels_v1.parquet

上述数据文件仅保存在本地，不上传GitHub。
GitHub仅保存代码、Notebook、设计文档和校验报告。

## 8. 步骤结论

步骤二已完成多维度中间表构建、字段设计、
数据一致性校验以及特征与标签分离。

下一步骤才进入探索性分析、滚动时间窗口特征
或模型训练数据准备。
"""

report_path.write_text(
    report_text,
    encoding="utf-8"
)

print("\n===== 步骤二报告生成结果 =====")
print("中间表汇总：", summary_csv_path)
print("设计与校验报告：", report_path)

,中间表,统计粒度,主键,行数,状态
0,user_summary,每名用户一行,user_id,10000,已完成
1,item_summary,每件商品一行,item_id,2876947,已完成
2,category_summary,每个品类一行,item_category,8916,已完成
3,time_hourly_summary,每小时一行,time,744,已完成
4,time_daily_summary,每天一行,action_date,31,已完成
5,time_weekly_summary,每周一行,week_start,5,已完成
6,user_item_features,每个历史用户—商品组合一行,user_id + item_id,4546934,已完成
7,user_item_labels,每个历史用户—商品组合一行,user_id + item_id,4546934,已完成，步骤三使用



===== 步骤二报告生成结果 =====
中间表汇总： ../reports/step2_intermediate_table_summary.csv
设计与校验报告： ../reports/intermediate_table_design.md


In [4]:
from pathlib import Path
import duckdb
import pandas as pd

# ============================================================
# 步骤三参数：固定观察期与标签期
# ============================================================

PROJECT_ROOT = Path("/Users/nyx/Desktop/user-behavior-analysis")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

SOURCE_PATH = (
    PROCESSED_DIR / "user_behavior_standardized_v1.parquet"
)

# 观察期：只能用这个时间点之前的数据计算特征
OBSERVATION_START = pd.Timestamp("2025-11-18 00:00:00")
CUTOFF_TIME = pd.Timestamp("2025-12-18 00:00:00")

# 标签期：判断用户在12月18日是否购买
LABEL_START = CUTOFF_TIME
LABEL_END = pd.Timestamp("2025-12-19 00:00:00")

assert SOURCE_PATH.exists(), f"找不到文件：{SOURCE_PATH}"

source_sql_path = SOURCE_PATH.as_posix()
cutoff_text = CUTOFF_TIME.strftime("%Y-%m-%d %H:%M:%S")
label_end_text = LABEL_END.strftime("%Y-%m-%d %H:%M:%S")

con = duckdb.connect()

# 检查观察期和标签期的记录数量
window_check = con.execute(f"""
SELECT
    COUNT(*) AS total_rows,

    COUNT(*) FILTER (
        WHERE time < TIMESTAMP '{cutoff_text}'
    ) AS observation_rows,

    COUNT(*) FILTER (
        WHERE time >= TIMESTAMP '{cutoff_text}'
          AND time < TIMESTAMP '{label_end_text}'
    ) AS label_rows,

    COUNT(*) FILTER (
        WHERE time >= TIMESTAMP '{cutoff_text}'
          AND time < TIMESTAMP '{label_end_text}'
          AND behavior_type = 4
    ) AS label_purchase_actions

FROM read_parquet('{source_sql_path}')
""").df()

# 检查标签期实际购买的用户—商品组合数
purchase_pair_check = con.execute(f"""
SELECT
    COUNT(*) AS label_purchase_pairs
FROM (
    SELECT DISTINCT
        user_id,
        item_id
    FROM read_parquet('{source_sql_path}')
    WHERE time >= TIMESTAMP '{cutoff_text}'
      AND time < TIMESTAMP '{label_end_text}'
      AND behavior_type = 4
)
""").df()

print("===== 步骤三：时间窗口定义 =====")
print("观察期开始：", OBSERVATION_START)
print("特征截止时间：", CUTOFF_TIME)
print("标签期开始：", LABEL_START)
print("标签期结束：", LABEL_END)

print("\n===== 时间窗口记录校验 =====")
display(window_check)

print("\n===== 标签期购买组合校验 =====")
display(purchase_pair_check)

# 与步骤二结果对账
assert int(window_check.loc[0, "total_rows"]) == 12_256_906
assert int(window_check.loc[0, "observation_rows"]) == 11_881_309
assert int(purchase_pair_check.loc[0, "label_purchase_pairs"]) == 3_151

print("\n时间窗口校验通过，未发现时间范围错误。")

===== 步骤三：时间窗口定义 =====
观察期开始： 2025-11-18 00:00:00
特征截止时间： 2025-12-18 00:00:00
标签期开始： 2025-12-18 00:00:00
标签期结束： 2025-12-19 00:00:00

===== 时间窗口记录校验 =====


,total_rows,observation_rows,label_rows,label_purchase_actions
0,12256906,11881309,375597,3586



===== 标签期购买组合校验 =====


,label_purchase_pairs
0,3151



时间窗口校验通过，未发现时间范围错误。


In [5]:
# ============================================================
# 步骤三：建立观察期数据视图和标签视图
# ============================================================

# 观察期明细：仅用于计算特征
con.execute(f"""
CREATE OR REPLACE TEMP VIEW observation_events AS
SELECT
    time,
    user_id,
    item_id,
    item_category,
    behavior_type
FROM read_parquet('{source_sql_path}')
WHERE time >= TIMESTAMP '2025-11-18 00:00:00'
  AND time < TIMESTAMP '{cutoff_text}'
""")

# 标签期购买组合：只用于生成预测标签
con.execute(f"""
CREATE OR REPLACE TEMP VIEW label_purchases AS
SELECT
    user_id,
    item_id,
    MIN(item_category) AS item_category,
    COUNT(*) AS label_purchase_actions,
    1 AS purchase_label
FROM read_parquet('{source_sql_path}')
WHERE time >= TIMESTAMP '{cutoff_text}'
  AND time < TIMESTAMP '{label_end_text}'
  AND behavior_type = 4
GROUP BY
    user_id,
    item_id
""")

# 观察期校验
observation_check = con.execute("""
SELECT
    COUNT(*) AS observation_rows,
    COUNT(DISTINCT user_id) AS observation_users,
    COUNT(DISTINCT item_id) AS observation_items,
    MIN(time) AS earliest_time,
    MAX(time) AS latest_time
FROM observation_events
""").df()

# 标签视图校验
label_check = con.execute("""
SELECT
    COUNT(*) AS label_purchase_pairs,
    COUNT(DISTINCT user_id) AS label_purchase_users,
    COUNT(DISTINCT item_id) AS label_purchase_items,
    SUM(label_purchase_actions) AS label_purchase_actions,
    SUM(label_purchase_actions - 1) AS repeated_purchase_actions
FROM label_purchases
""").df()

print("===== 观察期数据视图校验 =====")
display(observation_check)

print("\n===== 标签期购买视图校验 =====")
display(label_check)

assert int(observation_check.loc[0, "observation_rows"]) == 11_881_309
assert observation_check.loc[0, "latest_time"] < CUTOFF_TIME
assert int(label_check.loc[0, "label_purchase_pairs"]) == 3_151
assert int(label_check.loc[0, "label_purchase_actions"]) == 3_586

print("\n观察期与标签期已严格隔离，未发现数据泄露。")

===== 观察期数据视图校验 =====


,observation_rows,observation_users,observation_items,earliest_time,latest_time
0,11881309,9988,2803933,2025-11-18,2025-12-17 23:00:00



===== 标签期购买视图校验 =====


,label_purchase_pairs,label_purchase_users,label_purchase_items,label_purchase_actions,repeated_purchase_actions
0,3151,1552,3111,3586.0,435.0



观察期与标签期已严格隔离，未发现数据泄露。


In [6]:
# ============================================================
# 步骤三：构建用户多时间窗口特征
# ============================================================

WINDOW_DAYS = [1, 3, 7, 14, 30]

BEHAVIOR_MAP = {
    1: "view",
    2: "favorite",
    3: "cart",
    4: "purchase"
}

user_feature_expressions = []

for days in WINDOW_DAYS:
    window_start = (
        CUTOFF_TIME - pd.Timedelta(days=days)
    ).strftime("%Y-%m-%d %H:%M:%S")

    window_condition = (
        f"time >= TIMESTAMP '{window_start}' "
        f"AND time < TIMESTAMP '{cutoff_text}'"
    )

    # 总行为次数
    user_feature_expressions.append(
        f"""
        COUNT(*) FILTER (
            WHERE {window_condition}
        ) AS user_total_actions_{days}d
        """
    )

    # 浏览、收藏、加购、购买次数
    for behavior_code, behavior_name in BEHAVIOR_MAP.items():
        user_feature_expressions.append(
            f"""
            COUNT(*) FILTER (
                WHERE {window_condition}
                  AND behavior_type = {behavior_code}
            ) AS user_{behavior_name}_count_{days}d
            """
        )

    # 不同商品数
    user_feature_expressions.append(
        f"""
        COUNT(DISTINCT item_id) FILTER (
            WHERE {window_condition}
        ) AS user_unique_items_{days}d
        """
    )

    # 不同品类数
    user_feature_expressions.append(
        f"""
        COUNT(DISTINCT item_category) FILTER (
            WHERE {window_condition}
        ) AS user_unique_categories_{days}d
        """
    )

    # 活跃天数
    user_feature_expressions.append(
        f"""
        COUNT(DISTINCT CAST(time AS DATE)) FILTER (
            WHERE {window_condition}
        ) AS user_active_days_{days}d
        """
    )

    # 购买行为占比
    user_feature_expressions.append(
        f"""
        COALESCE(
            100.0 * COUNT(*) FILTER (
                WHERE {window_condition}
                  AND behavior_type = 4
            )
            / NULLIF(
                COUNT(*) FILTER (
                    WHERE {window_condition}
                ),
                0
            ),
            0
        ) AS user_purchase_action_pct_{days}d
        """
    )

    # 高意向行为占比：收藏、加购和购买
    user_feature_expressions.append(
        f"""
        COALESCE(
            100.0 * COUNT(*) FILTER (
                WHERE {window_condition}
                  AND behavior_type IN (2, 3, 4)
            )
            / NULLIF(
                COUNT(*) FILTER (
                    WHERE {window_condition}
                ),
                0
            ),
            0
        ) AS user_high_intent_action_pct_{days}d
        """
    )

user_feature_sql = ",\n".join(user_feature_expressions)

con.execute(f"""
CREATE OR REPLACE TEMP TABLE user_time_features AS
SELECT
    user_id,

    MIN(time) AS user_first_action_time,
    MAX(time) AS user_last_action_time,

    DATE_DIFF(
        'hour',
        MAX(time),
        TIMESTAMP '{cutoff_text}'
    ) AS user_recency_hours,

    {user_feature_sql}

FROM observation_events
GROUP BY user_id
""")

# ============================================================
# 用户特征校验
# ============================================================

user_feature_check = con.execute("""
SELECT
    COUNT(*) AS user_feature_rows,

    SUM(user_total_actions_30d)
        AS restored_observation_actions,

    MIN(user_recency_hours)
        AS min_user_recency_hours,

    MAX(user_recency_hours)
        AS max_user_recency_hours,

    SUM(
        CASE
            WHEN user_total_actions_1d > user_total_actions_3d
              OR user_total_actions_3d > user_total_actions_7d
              OR user_total_actions_7d > user_total_actions_14d
              OR user_total_actions_14d > user_total_actions_30d
            THEN 1
            ELSE 0
        END
    ) AS window_order_violations,

    SUM(
        CASE
            WHEN user_total_actions_30d !=
                 user_view_count_30d
               + user_favorite_count_30d
               + user_cart_count_30d
               + user_purchase_count_30d
            THEN 1
            ELSE 0
        END
    ) AS behavior_sum_violations

FROM user_time_features
""").df()

user_feature_schema = con.execute("""
DESCRIBE user_time_features
""").df()

top_active_users = con.execute("""
SELECT *
FROM user_time_features
ORDER BY user_total_actions_30d DESC
LIMIT 10
""").df()

print("===== 用户多时间窗口特征校验 =====")
display(user_feature_check)

print("\n用户特征字段数量：", len(user_feature_schema))
print("用户特征行数：", len(top_active_users), "（以下仅预览最活跃10人）")

display(top_active_users)

# 自动校验
assert int(user_feature_check.loc[0, "user_feature_rows"]) == 9_988
assert int(
    user_feature_check.loc[0, "restored_observation_actions"]
) == 11_881_309
assert int(
    user_feature_check.loc[0, "window_order_violations"]
) == 0
assert int(
    user_feature_check.loc[0, "behavior_sum_violations"]
) == 0

print("\n用户多时间窗口特征构建成功，全部校验通过。")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

===== 用户多时间窗口特征校验 =====


,user_feature_rows,restored_observation_actions,min_user_recency_hours,max_user_recency_hours,window_order_violations,behavior_sum_violations
0,9988,11881309.0,1,719,0.0,0.0



用户特征字段数量： 54
用户特征行数： 10 （以下仅预览最活跃10人）


,user_id,user_first_action_time,user_last_action_time,user_recency_hours,user_total_actions_1d,user_view_count_1d,user_favorite_count_1d,user_cart_count_1d,user_purchase_count_1d,user_unique_items_1d,...,user_total_actions_30d,user_view_count_30d,user_favorite_count_30d,user_cart_count_30d,user_purchase_count_30d,user_unique_items_30d,user_unique_categories_30d,user_active_days_30d,user_purchase_action_pct_30d,user_high_intent_action_pct_30d
0,36233277,2025-11-18 07:00:00,2025-12-17 21:00:00,3,779,682,94,1,2,296,...,30526,27263,2889,327,47,11395,372,30,0.153967,10.689249
1,65645933,2025-11-18 03:00:00,2025-12-17 23:00:00,1,177,155,22,0,0,81,...,20678,18037,2595,27,19,7204,324,29,0.091885,12.772028
2,73196588,2025-11-18 00:00:00,2025-12-17 23:00:00,1,369,369,0,0,0,305,...,19643,19643,0,0,0,13704,1114,30,0.000000,0.000000
3,130270245,2025-11-18 00:00:00,2025-12-17 23:00:00,1,1012,987,3,14,8,323,...,19313,18392,718,185,18,5527,403,27,0.093201,4.768809
4,59511789,2025-11-18 10:00:00,2025-12-17 23:00:00,1,850,780,57,13,0,401,...,19148,17649,1219,246,34,6411,308,30,0.177564,7.828494
5,7234861,2025-11-18 00:00:00,2025-12-17 23:00:00,1,572,494,71,7,0,220,...,17210,15461,1134,566,49,6005,605,30,0.284718,10.162696
6,83813302,2025-11-18 09:00:00,2025-12-17 23:00:00,1,681,663,10,6,2,251,...,17026,16656,274,80,16,6674,308,30,0.093974,2.173147
7,52577851,2025-11-18 08:00:00,2025-12-17 23:00:00,1,374,351,0,23,0,154,...,13388,12525,10,791,62,4747,266,30,0.463101,6.446071
8,137175187,2025-11-18 01:00:00,2025-12-17 17:00:00,7,37,36,1,0,0,12,...,12744,12186,243,284,31,4239,213,30,0.243252,4.378531
9,5315279,2025-11-18 06:00:00,2025-12-17 23:00:00,1,676,577,1,95,3,263,...,12281,10284,168,1797,32,4284,281,26,0.260565,16.260891



用户多时间窗口特征构建成功，全部校验通过。


In [7]:
# ============================================================
# 步骤三：构建商品多时间窗口特征
# ============================================================

ITEM_WINDOW_DAYS = [1, 7, 30]

item_feature_expressions = []

for days in ITEM_WINDOW_DAYS:
    window_start = (
        CUTOFF_TIME - pd.Timedelta(days=days)
    ).strftime("%Y-%m-%d %H:%M:%S")

    window_condition = (
        f"time >= TIMESTAMP '{window_start}' "
        f"AND time < TIMESTAMP '{cutoff_text}'"
    )

    # 商品总行为次数
    item_feature_expressions.append(
        f"""
        COUNT(*) FILTER (
            WHERE {window_condition}
        ) AS item_total_actions_{days}d
        """
    )

    # 商品浏览、收藏、加购和购买次数
    for behavior_code, behavior_name in BEHAVIOR_MAP.items():
        item_feature_expressions.append(
            f"""
            COUNT(*) FILTER (
                WHERE {window_condition}
                  AND behavior_type = {behavior_code}
            ) AS item_{behavior_name}_count_{days}d
            """
        )

    # 商品交互用户数
    item_feature_expressions.append(
        f"""
        COUNT(DISTINCT user_id) FILTER (
            WHERE {window_condition}
        ) AS item_unique_users_{days}d
        """
    )

    # 商品购买用户数
    item_feature_expressions.append(
        f"""
        COUNT(DISTINCT user_id) FILTER (
            WHERE {window_condition}
              AND behavior_type = 4
        ) AS item_unique_buyers_{days}d
        """
    )

    # 商品购买行为占比
    item_feature_expressions.append(
        f"""
        COALESCE(
            100.0 * COUNT(*) FILTER (
                WHERE {window_condition}
                  AND behavior_type = 4
            )
            / NULLIF(
                COUNT(*) FILTER (
                    WHERE {window_condition}
                ),
                0
            ),
            0
        ) AS item_purchase_action_pct_{days}d
        """
    )

    # 商品高意向行为占比
    item_feature_expressions.append(
        f"""
        COALESCE(
            100.0 * COUNT(*) FILTER (
                WHERE {window_condition}
                  AND behavior_type IN (2, 3, 4)
            )
            / NULLIF(
                COUNT(*) FILTER (
                    WHERE {window_condition}
                ),
                0
            ),
            0
        ) AS item_high_intent_action_pct_{days}d
        """
    )

item_feature_sql = ",\n".join(item_feature_expressions)

con.execute(f"""
CREATE OR REPLACE TEMP TABLE item_time_features AS
SELECT
    item_id,

    MIN(item_category) AS item_category,
    COUNT(DISTINCT item_category) AS item_category_count,

    MIN(time) AS item_first_action_time,
    MAX(time) AS item_last_action_time,

    DATE_DIFF(
        'hour',
        MAX(time),
        TIMESTAMP '{cutoff_text}'
    ) AS item_recency_hours,

    {item_feature_sql}

FROM observation_events
GROUP BY item_id
""")

# ============================================================
# 商品特征校验
# ============================================================

item_feature_check = con.execute("""
SELECT
    COUNT(*) AS item_feature_rows,

    SUM(item_total_actions_30d)
        AS restored_observation_actions,

    MIN(item_recency_hours)
        AS min_item_recency_hours,

    MAX(item_recency_hours)
        AS max_item_recency_hours,

    SUM(
        CASE
            WHEN item_category_count != 1
            THEN 1
            ELSE 0
        END
    ) AS multiple_category_items,

    SUM(
        CASE
            WHEN item_total_actions_1d > item_total_actions_7d
              OR item_total_actions_7d > item_total_actions_30d
            THEN 1
            ELSE 0
        END
    ) AS window_order_violations,

    SUM(
        CASE
            WHEN item_total_actions_30d !=
                 item_view_count_30d
               + item_favorite_count_30d
               + item_cart_count_30d
               + item_purchase_count_30d
            THEN 1
            ELSE 0
        END
    ) AS behavior_sum_violations

FROM item_time_features
""").df()

item_feature_schema = con.execute("""
DESCRIBE item_time_features
""").df()

top_active_items = con.execute("""
SELECT
    item_id,
    item_category,
    item_recency_hours,
    item_total_actions_1d,
    item_total_actions_7d,
    item_total_actions_30d,
    item_view_count_30d,
    item_favorite_count_30d,
    item_cart_count_30d,
    item_purchase_count_30d,
    item_unique_users_30d,
    item_unique_buyers_30d,
    item_purchase_action_pct_30d
FROM item_time_features
ORDER BY item_total_actions_30d DESC
LIMIT 10
""").df()

print("===== 商品多时间窗口特征校验 =====")
display(item_feature_check)

print("\n商品特征字段数量：", len(item_feature_schema))
print("以下仅预览观察期行为量最高的10件商品：")
display(top_active_items)

# 自动校验
assert int(item_feature_check.loc[0, "item_feature_rows"]) == 2_803_933
assert int(
    item_feature_check.loc[0, "restored_observation_actions"]
) == 11_881_309
assert int(
    item_feature_check.loc[0, "multiple_category_items"]
) == 0
assert int(
    item_feature_check.loc[0, "window_order_violations"]
) == 0
assert int(
    item_feature_check.loc[0, "behavior_sum_violations"]
) == 0

print("\n商品多时间窗口特征构建成功，全部校验通过。")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

===== 商品多时间窗口特征校验 =====


,item_feature_rows,restored_observation_actions,min_item_recency_hours,max_item_recency_hours,multiple_category_items,window_order_violations,behavior_sum_violations
0,2803933,11881309.0,1,720,0.0,0.0,0.0



商品特征字段数量： 33
以下仅预览观察期行为量最高的10件商品：


,item_id,item_category,item_recency_hours,item_total_actions_1d,item_total_actions_7d,item_total_actions_30d,item_view_count_30d,item_favorite_count_30d,item_cart_count_30d,item_purchase_count_30d,item_unique_users_30d,item_unique_buyers_30d,item_purchase_action_pct_30d
0,112921337,6000,14,1,364,1441,1427,5,8,1,516,1,0.069396
1,97655171,6000,2,34,347,1267,1228,7,19,13,330,12,1.026046
2,387911330,6000,2,30,240,1073,1024,10,20,19,290,15,1.770736
3,135104537,6000,4,24,156,910,895,7,8,0,331,0,0.000000
4,14087919,5232,4,20,171,796,713,15,33,35,199,25,4.396985
5,5685392,3381,123,0,7,793,767,12,12,2,302,2,0.252207
6,277922302,5399,4,7,72,791,746,24,20,1,270,1,0.126422
7,128186279,3381,1,9,83,788,758,13,16,1,338,1,0.126904
8,2217535,6000,1,15,314,782,770,1,10,1,307,1,0.127877
9,209323160,10072,2,6,522,744,716,2,26,0,282,0,0.000000



商品多时间窗口特征构建成功，全部校验通过。


In [8]:
# ============================================================
# 步骤三：构建品类多时间窗口特征
# ============================================================

CATEGORY_WINDOW_DAYS = [1, 3, 7, 14, 30]

category_feature_expressions = []

for days in CATEGORY_WINDOW_DAYS:
    window_start = (
        CUTOFF_TIME - pd.Timedelta(days=days)
    ).strftime("%Y-%m-%d %H:%M:%S")

    window_condition = (
        f"time >= TIMESTAMP '{window_start}' "
        f"AND time < TIMESTAMP '{cutoff_text}'"
    )

    # 品类总行为次数
    category_feature_expressions.append(
        f"""
        COUNT(*) FILTER (
            WHERE {window_condition}
        ) AS category_total_actions_{days}d
        """
    )

    # 品类浏览、收藏、加购和购买次数
    for behavior_code, behavior_name in BEHAVIOR_MAP.items():
        category_feature_expressions.append(
            f"""
            COUNT(*) FILTER (
                WHERE {window_condition}
                  AND behavior_type = {behavior_code}
            ) AS category_{behavior_name}_count_{days}d
            """
        )

    # 品类交互用户数
    category_feature_expressions.append(
        f"""
        COUNT(DISTINCT user_id) FILTER (
            WHERE {window_condition}
        ) AS category_unique_users_{days}d
        """
    )

    # 品类商品数
    category_feature_expressions.append(
        f"""
        COUNT(DISTINCT item_id) FILTER (
            WHERE {window_condition}
        ) AS category_unique_items_{days}d
        """
    )

    # 品类购买用户数
    category_feature_expressions.append(
        f"""
        COUNT(DISTINCT user_id) FILTER (
            WHERE {window_condition}
              AND behavior_type = 4
        ) AS category_unique_buyers_{days}d
        """
    )

    # 品类购买行为占比
    category_feature_expressions.append(
        f"""
        COALESCE(
            100.0 * COUNT(*) FILTER (
                WHERE {window_condition}
                  AND behavior_type = 4
            )
            / NULLIF(
                COUNT(*) FILTER (
                    WHERE {window_condition}
                ),
                0
            ),
            0
        ) AS category_purchase_action_pct_{days}d
        """
    )

    # 品类高意向行为占比
    category_feature_expressions.append(
        f"""
        COALESCE(
            100.0 * COUNT(*) FILTER (
                WHERE {window_condition}
                  AND behavior_type IN (2, 3, 4)
            )
            / NULLIF(
                COUNT(*) FILTER (
                    WHERE {window_condition}
                ),
                0
            ),
            0
        ) AS category_high_intent_action_pct_{days}d
        """
    )

category_feature_sql = ",\n".join(category_feature_expressions)

con.execute(f"""
CREATE OR REPLACE TEMP TABLE category_time_features AS
SELECT
    item_category,

    MIN(time) AS category_first_action_time,
    MAX(time) AS category_last_action_time,

    DATE_DIFF(
        'hour',
        MAX(time),
        TIMESTAMP '{cutoff_text}'
    ) AS category_recency_hours,

    {category_feature_sql}

FROM observation_events
GROUP BY item_category
""")

# ============================================================
# 品类特征校验
# ============================================================

expected_category_rows = con.execute("""
SELECT COUNT(DISTINCT item_category)
FROM observation_events
""").fetchone()[0]

category_feature_check = con.execute("""
SELECT
    COUNT(*) AS category_feature_rows,

    SUM(category_total_actions_30d)
        AS restored_observation_actions,

    SUM(category_unique_items_30d)
        AS restored_observation_items,

    MIN(category_recency_hours)
        AS min_category_recency_hours,

    MAX(category_recency_hours)
        AS max_category_recency_hours,

    SUM(
        CASE
            WHEN category_total_actions_1d >
                 category_total_actions_3d
              OR category_total_actions_3d >
                 category_total_actions_7d
              OR category_total_actions_7d >
                 category_total_actions_14d
              OR category_total_actions_14d >
                 category_total_actions_30d
            THEN 1
            ELSE 0
        END
    ) AS window_order_violations,

    SUM(
        CASE
            WHEN category_total_actions_30d !=
                 category_view_count_30d
               + category_favorite_count_30d
               + category_cart_count_30d
               + category_purchase_count_30d
            THEN 1
            ELSE 0
        END
    ) AS behavior_sum_violations

FROM category_time_features
""").df()

category_feature_schema = con.execute("""
DESCRIBE category_time_features
""").df()

top_active_categories = con.execute("""
SELECT
    item_category,
    category_recency_hours,
    category_total_actions_1d,
    category_total_actions_3d,
    category_total_actions_7d,
    category_total_actions_14d,
    category_total_actions_30d,
    category_view_count_30d,
    category_favorite_count_30d,
    category_cart_count_30d,
    category_purchase_count_30d,
    category_unique_users_30d,
    category_unique_items_30d,
    category_unique_buyers_30d,
    category_purchase_action_pct_30d
FROM category_time_features
ORDER BY category_total_actions_30d DESC
LIMIT 10
""").df()

print("===== 品类多时间窗口特征校验 =====")
display(category_feature_check)

print("\n品类特征字段数量：", len(category_feature_schema))
print("以下仅预览观察期行为量最高的10个品类：")
display(top_active_categories)

# 自动校验
assert int(
    category_feature_check.loc[0, "category_feature_rows"]
) == int(expected_category_rows)

assert int(
    category_feature_check.loc[0, "restored_observation_actions"]
) == 11_881_309

assert int(
    category_feature_check.loc[0, "restored_observation_items"]
) == 2_803_933

assert int(
    category_feature_check.loc[0, "window_order_violations"]
) == 0

assert int(
    category_feature_check.loc[0, "behavior_sum_violations"]
) == 0

print("\n品类多时间窗口特征构建成功，全部校验通过。")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

===== 品类多时间窗口特征校验 =====


,category_feature_rows,restored_observation_actions,restored_observation_items,min_category_recency_hours,max_category_recency_hours,window_order_violations,behavior_sum_violations
0,8875,11881309.0,2803933.0,1,720,0.0,0.0



品类特征字段数量： 54
以下仅预览观察期行为量最高的10个品类：


,item_category,category_recency_hours,category_total_actions_1d,category_total_actions_3d,category_total_actions_7d,category_total_actions_14d,category_total_actions_30d,category_view_count_30d,category_favorite_count_30d,category_cart_count_30d,category_purchase_count_30d,category_unique_users_30d,category_unique_items_30d,category_unique_buyers_30d,category_purchase_action_pct_30d
0,1863,1,12519,36386,101667,194479,380133,359242,9880,9068,1943,5932,81267,1229,0.511137
1,13230,1,10153,30363,82636,161481,348139,334383,7061,5873,822,5567,72630,555,0.236113
2,5027,1,12124,37191,100329,185935,322485,309519,6786,5368,812,5443,58819,609,0.251795
3,5894,1,9588,30715,81716,153526,320336,305309,7658,6440,929,5247,81728,593,0.290008
4,6513,1,9491,28456,76524,140286,287091,273069,6501,6490,1031,5235,63441,683,0.359120
5,5399,1,9435,29585,79176,149135,272391,259664,6434,5254,1039,5261,53016,644,0.381437
6,11279,1,5380,16840,47913,90056,181980,173083,4560,3630,707,4652,46294,575,0.388504
7,2825,1,4411,14314,39244,73502,158693,151232,3265,3587,609,4518,42838,407,0.383760
8,5232,1,4528,13742,36947,70186,139360,130906,2514,4364,1576,4911,25153,935,1.130884
9,10894,1,3873,11872,32833,64023,134565,127224,2722,3777,842,4472,32658,580,0.625720



品类多时间窗口特征构建成功，全部校验通过。


In [9]:
# ============================================================
# 检查只在标签日出现的41个冷启动品类
# ============================================================

cold_start_categories = con.execute(f"""
WITH observation_categories AS (
    SELECT DISTINCT item_category
    FROM observation_events
)

SELECT
    label_data.item_category,

    COUNT(*) AS label_total_actions,

    COUNT(*) FILTER (
        WHERE behavior_type = 1
    ) AS label_view_count,

    COUNT(*) FILTER (
        WHERE behavior_type = 2
    ) AS label_favorite_count,

    COUNT(*) FILTER (
        WHERE behavior_type = 3
    ) AS label_cart_count,

    COUNT(*) FILTER (
        WHERE behavior_type = 4
    ) AS label_purchase_count,

    COUNT(DISTINCT user_id) AS label_unique_users,
    COUNT(DISTINCT item_id) AS label_unique_items,

    MIN(time) AS first_appearance_time,
    MAX(time) AS last_appearance_time

FROM read_parquet('{source_sql_path}') AS label_data

WHERE time >= TIMESTAMP '{cutoff_text}'
  AND time < TIMESTAMP '{label_end_text}'

  AND item_category NOT IN (
      SELECT item_category
      FROM observation_categories
  )

GROUP BY label_data.item_category
ORDER BY label_total_actions DESC
""").df()

cold_start_summary = con.execute(f"""
WITH observation_categories AS (
    SELECT DISTINCT item_category
    FROM observation_events
),

cold_start_events AS (
    SELECT *
    FROM read_parquet('{source_sql_path}')
    WHERE time >= TIMESTAMP '{cutoff_text}'
      AND time < TIMESTAMP '{label_end_text}'
      AND item_category NOT IN (
          SELECT item_category
          FROM observation_categories
      )
)

SELECT
    COUNT(DISTINCT item_category)
        AS cold_start_categories,

    COUNT(*) AS cold_start_actions,

    COUNT(DISTINCT user_id)
        AS cold_start_users,

    COUNT(DISTINCT item_id)
        AS cold_start_items,

    COUNT(*) FILTER (
        WHERE behavior_type = 4
    ) AS cold_start_purchase_actions,

    COUNT(DISTINCT item_category) FILTER (
        WHERE behavior_type = 4
    ) AS purchased_cold_start_categories

FROM cold_start_events
""").df()

print("===== 冷启动品类总体情况 =====")
display(cold_start_summary)

print("\n===== 41个冷启动品类明细 =====")
display(cold_start_categories)

assert len(cold_start_categories) == 41

print("\n确认：这41个品类仅在标签日出现，没有进入观察期特征。")

===== 冷启动品类总体情况 =====


,cold_start_categories,cold_start_actions,cold_start_users,cold_start_items,cold_start_purchase_actions,purchased_cold_start_categories
0,41,109,39,51,0,0



===== 41个冷启动品类明细 =====


,item_category,label_total_actions,label_view_count,label_favorite_count,label_cart_count,label_purchase_count,label_unique_users,label_unique_items,first_appearance_time,last_appearance_time
0,5045,8,8,0,0,0,1,8,2025-12-18 07:00:00,2025-12-18 07:00:00
1,2394,8,7,0,1,0,1,1,2025-12-18 21:00:00,2025-12-18 21:00:00
2,13871,6,6,0,0,0,2,1,2025-12-18 15:00:00,2025-12-18 15:00:00
3,8175,5,4,0,1,0,1,1,2025-12-18 13:00:00,2025-12-18 13:00:00
4,4403,4,4,0,0,0,1,1,2025-12-18 11:00:00,2025-12-18 11:00:00
5,12843,4,4,0,0,0,1,2,2025-12-18 01:00:00,2025-12-18 01:00:00
6,2563,4,4,0,0,0,1,1,2025-12-18 12:00:00,2025-12-18 12:00:00
7,13882,4,3,1,0,0,1,1,2025-12-18 10:00:00,2025-12-18 22:00:00
8,741,4,4,0,0,0,1,2,2025-12-18 20:00:00,2025-12-18 20:00:00
9,7302,3,3,0,0,0,1,1,2025-12-18 21:00:00,2025-12-18 21:00:00



确认：这41个品类仅在标签日出现，没有进入观察期特征。


In [10]:
# ============================================================
# 步骤三：构建用户—品类偏好多时间窗口特征
# ============================================================

USER_CATEGORY_WINDOW_DAYS = [1, 7, 30]

user_category_expressions = []

for days in USER_CATEGORY_WINDOW_DAYS:
    window_start = (
        CUTOFF_TIME - pd.Timedelta(days=days)
    ).strftime("%Y-%m-%d %H:%M:%S")

    window_condition = (
        f"time >= TIMESTAMP '{window_start}' "
        f"AND time < TIMESTAMP '{cutoff_text}'"
    )

    # 总行为数
    user_category_expressions.append(
        f"""
        COUNT(*) FILTER (
            WHERE {window_condition}
        ) AS user_category_total_actions_{days}d
        """
    )

    # 浏览、收藏、加购、购买次数
    for behavior_code, behavior_name in BEHAVIOR_MAP.items():
        user_category_expressions.append(
            f"""
            COUNT(*) FILTER (
                WHERE {window_condition}
                  AND behavior_type = {behavior_code}
            ) AS user_category_{behavior_name}_count_{days}d
            """
        )

    # 交互商品数量
    user_category_expressions.append(
        f"""
        COUNT(DISTINCT item_id) FILTER (
            WHERE {window_condition}
        ) AS user_category_unique_items_{days}d
        """
    )

user_category_sql = ",\n".join(user_category_expressions)

# 第一部分：基础聚合
con.execute(f"""
CREATE OR REPLACE TEMP TABLE user_category_features_base AS
SELECT
    user_id,
    item_category,

    MIN(time) AS user_category_first_action_time,
    MAX(time) AS user_category_last_action_time,

    DATE_DIFF(
        'hour',
        MAX(time),
        TIMESTAMP '{cutoff_text}'
    ) AS user_category_recency_hours,

    {user_category_sql}

FROM observation_events

GROUP BY
    user_id,
    item_category
""")

# 第二部分：计算用户品类偏好占比和排名
con.execute("""
CREATE OR REPLACE TEMP TABLE user_category_features AS
SELECT
    uc.*,

    COALESCE(
        100.0 * uc.user_category_total_actions_1d
        / NULLIF(u.user_total_actions_1d, 0),
        0
    ) AS user_category_action_share_1d,

    COALESCE(
        100.0 * uc.user_category_total_actions_7d
        / NULLIF(u.user_total_actions_7d, 0),
        0
    ) AS user_category_action_share_7d,

    COALESCE(
        100.0 * uc.user_category_total_actions_30d
        / NULLIF(u.user_total_actions_30d, 0),
        0
    ) AS user_category_action_share_30d,

    ROW_NUMBER() OVER (
        PARTITION BY uc.user_id
        ORDER BY
            uc.user_category_total_actions_7d DESC,
            uc.user_category_total_actions_30d DESC,
            uc.user_category_recency_hours ASC,
            uc.item_category
    ) AS user_category_preference_rank

FROM user_category_features_base AS uc

LEFT JOIN user_time_features AS u
    ON uc.user_id = u.user_id
""")

# ============================================================
# 校验
# ============================================================

expected_user_category_rows = con.execute("""
SELECT COUNT(*)
FROM (
    SELECT DISTINCT
        user_id,
        item_category
    FROM observation_events
)
""").fetchone()[0]

user_category_check = con.execute("""
SELECT
    COUNT(*) AS user_category_rows,

    COUNT(DISTINCT user_id)
        AS covered_users,

    COUNT(DISTINCT item_category)
        AS covered_categories,

    SUM(user_category_total_actions_30d)
        AS restored_observation_actions,

    MIN(user_category_recency_hours)
        AS min_recency_hours,

    MAX(user_category_recency_hours)
        AS max_recency_hours,

    SUM(
        CASE
            WHEN user_category_total_actions_1d >
                 user_category_total_actions_7d
              OR user_category_total_actions_7d >
                 user_category_total_actions_30d
            THEN 1
            ELSE 0
        END
    ) AS window_order_violations,

    SUM(
        CASE
            WHEN user_category_total_actions_30d !=
                 user_category_view_count_30d
               + user_category_favorite_count_30d
               + user_category_cart_count_30d
               + user_category_purchase_count_30d
            THEN 1
            ELSE 0
        END
    ) AS behavior_sum_violations

FROM user_category_features
""").df()

user_category_schema = con.execute("""
DESCRIBE user_category_features
""").df()

top_user_category_pairs = con.execute("""
SELECT
    user_id,
    item_category,
    user_category_preference_rank,
    user_category_recency_hours,
    user_category_total_actions_1d,
    user_category_total_actions_7d,
    user_category_total_actions_30d,
    user_category_view_count_30d,
    user_category_favorite_count_30d,
    user_category_cart_count_30d,
    user_category_purchase_count_30d,
    user_category_unique_items_30d,
    user_category_action_share_30d
FROM user_category_features
ORDER BY user_category_total_actions_30d DESC
LIMIT 10
""").df()

print("===== 用户—品类偏好特征校验 =====")
display(user_category_check)

print("\n用户—品类特征字段数量：", len(user_category_schema))
print("以下仅预览行为量最高的10个用户—品类组合：")
display(top_user_category_pairs)

# 自动检查
assert int(
    user_category_check.loc[0, "user_category_rows"]
) == int(expected_user_category_rows)

assert int(
    user_category_check.loc[0, "covered_users"]
) == 9_988

assert int(
    user_category_check.loc[0, "restored_observation_actions"]
) == 11_881_309

assert int(
    user_category_check.loc[0, "window_order_violations"]
) == 0

assert int(
    user_category_check.loc[0, "behavior_sum_violations"]
) == 0

print("\n用户—品类偏好特征构建成功，全部校验通过。")


===== 用户—品类偏好特征校验 =====


,user_category_rows,covered_users,covered_categories,restored_observation_actions,min_recency_hours,max_recency_hours,window_order_violations,behavior_sum_violations
0,880669,9988,8875,11881309.0,1,720,0.0,0.0



用户—品类特征字段数量： 27
以下仅预览行为量最高的10个用户—品类组合：


,user_id,item_category,user_category_preference_rank,user_category_recency_hours,user_category_total_actions_1d,user_category_total_actions_7d,user_category_total_actions_30d,user_category_view_count_30d,user_category_favorite_count_30d,user_category_cart_count_30d,user_category_purchase_count_30d,user_category_unique_items_30d,user_category_action_share_30d
0,123842164,6977,1,1,246,1369,3998,3103,5,371,519,843,33.366717
1,91496013,13230,1,4,5,135,3871,3817,52,0,2,1155,48.321059
2,36233277,5399,2,4,39,765,3816,3354,444,18,0,1355,12.500819
3,65645933,5894,3,1,45,288,3445,3019,424,1,1,1125,16.660219
4,38839327,13230,1,8,299,505,3051,2912,27,107,5,944,25.957121
5,113283932,1121,5,487,0,0,3035,3035,0,0,0,591,33.183906
6,65645933,1863,2,1,27,290,2926,2490,434,1,1,943,14.150305
7,53080199,13230,1,1,42,524,2909,2821,32,50,6,1029,24.761662
8,65645933,5027,1,22,38,437,2604,2285,315,2,2,696,12.593094
9,36233277,5689,1,4,127,1358,2576,2212,338,26,0,877,8.438708



用户—品类偏好特征构建成功，全部校验通过。


In [11]:
# ============================================================
# 步骤三：保存已完成的特征表检查点
# ============================================================

STEP3_FEATURE_DIR = PROCESSED_DIR / "step3_features"
STEP3_FEATURE_DIR.mkdir(parents=True, exist_ok=True)

feature_exports = {
    "user_time_features":
        STEP3_FEATURE_DIR / "user_time_features_v1.parquet",

    "item_time_features":
        STEP3_FEATURE_DIR / "item_time_features_v1.parquet",

    "category_time_features":
        STEP3_FEATURE_DIR / "category_time_features_v1.parquet",

    "user_category_features":
        STEP3_FEATURE_DIR / "user_category_features_v1.parquet"
}

export_results = []

for table_name, output_path in feature_exports.items():

    # 只覆盖本步骤生成的同名检查点
    if output_path.exists():
        output_path.unlink()

    output_sql_path = output_path.as_posix()

    con.execute(f"""
    COPY (
        SELECT *
        FROM {table_name}
    )
    TO '{output_sql_path}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
    """)

    saved_rows = con.execute(f"""
    SELECT COUNT(*)
    FROM read_parquet('{output_sql_path}')
    """).fetchone()[0]

    file_size_mb = output_path.stat().st_size / 1024 / 1024

    export_results.append({
        "feature_table": table_name,
        "saved_rows": saved_rows,
        "file_size_mb": round(file_size_mb, 2),
        "output_file": output_path.name
    })

feature_export_check = pd.DataFrame(export_results)

print("===== 步骤三特征检查点保存结果 =====")
display(feature_export_check)

# 自动校验
expected_rows = {
    "user_time_features": 9_988,
    "item_time_features": 2_803_933,
    "category_time_features": 8_875,
    "user_category_features": 880_669
}

for _, row in feature_export_check.iterrows():
    assert int(row["saved_rows"]) == expected_rows[row["feature_table"]]

print("\n四张特征表已保存为Parquet，记录数全部一致。")
print("保存目录：", STEP3_FEATURE_DIR)

===== 步骤三特征检查点保存结果 =====


,feature_table,saved_rows,file_size_mb,output_file
0,user_time_features,9988,0.79,user_time_features_v1.parquet
1,item_time_features,2803933,34.73,item_time_features_v1.parquet
2,category_time_features,8875,0.55,category_time_features_v1.parquet
3,user_category_features,880669,10.54,user_category_features_v1.parquet



四张特征表已保存为Parquet，记录数全部一致。
保存目录： /Users/nyx/Desktop/user-behavior-analysis/data/processed/step3_features


In [12]:
# ============================================================
# 步骤三：构建历史用户—商品候选特征
# ============================================================

USER_ITEM_WINDOW_DAYS = [1, 3, 7, 14, 30]

user_item_expressions = []

for days in USER_ITEM_WINDOW_DAYS:
    window_start = (
        CUTOFF_TIME - pd.Timedelta(days=days)
    ).strftime("%Y-%m-%d %H:%M:%S")

    window_condition = (
        f"time >= TIMESTAMP '{window_start}' "
        f"AND time < TIMESTAMP '{cutoff_text}'"
    )

    # 用户—商品总交互次数
    user_item_expressions.append(
        f"""
        COUNT(*) FILTER (
            WHERE {window_condition}
        ) AS pair_total_actions_{days}d
        """
    )

    # 浏览、收藏、加购和购买次数
    for behavior_code, behavior_name in BEHAVIOR_MAP.items():
        user_item_expressions.append(
            f"""
            COUNT(*) FILTER (
                WHERE {window_condition}
                  AND behavior_type = {behavior_code}
            ) AS pair_{behavior_name}_count_{days}d
            """
        )

user_item_sql = ",\n".join(user_item_expressions)

con.execute(f"""
CREATE OR REPLACE TEMP TABLE user_item_time_features AS
SELECT
    user_id,
    item_id,
    MIN(item_category) AS item_category,

    MIN(time) AS pair_first_action_time,
    MAX(time) AS pair_last_action_time,

    DATE_DIFF(
        'hour',
        MAX(time),
        TIMESTAMP '{cutoff_text}'
    ) AS pair_recency_hours,

    {user_item_sql},

    COUNT(DISTINCT CAST(time AS DATE)) FILTER (
        WHERE time >= TIMESTAMP '{cutoff_text}'
                         - INTERVAL '7 days'
    ) AS pair_active_days_7d,

    COUNT(DISTINCT CAST(time AS DATE)) AS pair_active_days_30d,

    COUNT(DISTINCT DATE_TRUNC('hour', time)) FILTER (
        WHERE time >= TIMESTAMP '{cutoff_text}'
                         - INTERVAL '7 days'
    ) AS pair_active_hours_7d,

    COUNT(DISTINCT DATE_TRUNC('hour', time))
        AS pair_active_hours_30d,

    GREATEST(
        COUNT(*) FILTER (
            WHERE behavior_type = 1
        ) - 1,
        0
    ) AS pair_repeat_view_count_30d,

    COALESCE(
        100.0 * COUNT(*) FILTER (
            WHERE time >= TIMESTAMP '{cutoff_text}'
                             - INTERVAL '7 days'
              AND behavior_type IN (2, 3, 4)
        )
        / NULLIF(
            COUNT(*) FILTER (
                WHERE time >= TIMESTAMP '{cutoff_text}'
                                 - INTERVAL '7 days'
            ),
            0
        ),
        0
    ) AS pair_high_intent_action_pct_7d,

    COALESCE(
        100.0 * COUNT(*) FILTER (
            WHERE behavior_type IN (2, 3, 4)
        )
        / NULLIF(COUNT(*), 0),
        0
    ) AS pair_high_intent_action_pct_30d

FROM observation_events

GROUP BY
    user_id,
    item_id
""")

# ============================================================
# 候选集合与标签覆盖率校验
# ============================================================

candidate_check = con.execute("""
SELECT
    COUNT(*) AS candidate_pairs,

    SUM(pair_total_actions_30d)
        AS restored_observation_actions,

    MIN(pair_recency_hours)
        AS min_pair_recency_hours,

    MAX(pair_recency_hours)
        AS max_pair_recency_hours,

    SUM(
        CASE
            WHEN pair_total_actions_1d >
                 pair_total_actions_3d
              OR pair_total_actions_3d >
                 pair_total_actions_7d
              OR pair_total_actions_7d >
                 pair_total_actions_14d
              OR pair_total_actions_14d >
                 pair_total_actions_30d
            THEN 1
            ELSE 0
        END
    ) AS window_order_violations,

    SUM(
        CASE
            WHEN pair_total_actions_30d !=
                 pair_view_count_30d
               + pair_favorite_count_30d
               + pair_cart_count_30d
               + pair_purchase_count_30d
            THEN 1
            ELSE 0
        END
    ) AS behavior_sum_violations

FROM user_item_time_features
""").df()

candidate_label_check = con.execute("""
SELECT
    COUNT(*) AS candidate_pairs,

    SUM(
        CASE
            WHEN lp.purchase_label = 1
            THEN 1
            ELSE 0
        END
    ) AS positive_labels,

    100.0 * SUM(
        CASE
            WHEN lp.purchase_label = 1
            THEN 1
            ELSE 0
        END
    ) / COUNT(*) AS positive_rate_pct,

    100.0 * SUM(
        CASE
            WHEN lp.purchase_label = 1
            THEN 1
            ELSE 0
        END
    ) / (
        SELECT COUNT(*)
        FROM label_purchases
    ) AS candidate_purchase_coverage_pct

FROM user_item_time_features AS pair

LEFT JOIN label_purchases AS lp
    ON pair.user_id = lp.user_id
   AND pair.item_id = lp.item_id
""").df()

user_item_schema = con.execute("""
DESCRIBE user_item_time_features
""").df()

print("===== 历史用户—商品候选特征校验 =====")
display(candidate_check)

print("\n===== 候选标签与覆盖率校验 =====")
display(candidate_label_check)

print("\n用户—商品候选特征字段数量：", len(user_item_schema))

# 自动校验
assert int(candidate_check.loc[0, "candidate_pairs"]) == 4_546_934

assert int(
    candidate_check.loc[0, "restored_observation_actions"]
) == 11_881_309

assert int(
    candidate_check.loc[0, "window_order_violations"]
) == 0

assert int(
    candidate_check.loc[0, "behavior_sum_violations"]
) == 0

assert int(
    candidate_label_check.loc[0, "positive_labels"]
) == 1_028

print("\n历史用户—商品候选特征构建成功，全部校验通过。")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

===== 历史用户—商品候选特征校验 =====


,candidate_pairs,restored_observation_actions,min_pair_recency_hours,max_pair_recency_hours,window_order_violations,behavior_sum_violations
0,4546934,11881309.0,1,720,0.0,0.0



===== 候选标签与覆盖率校验 =====


,candidate_pairs,positive_labels,positive_rate_pct,candidate_purchase_coverage_pct
0,4546934,1028.0,0.022609,32.624564



用户—商品候选特征字段数量： 38

历史用户—商品候选特征构建成功，全部校验通过。


In [13]:
# ============================================================
# 步骤三：保存用户—商品特征与独立标签
# ============================================================

PAIR_FEATURE_PATH = (
    STEP3_FEATURE_DIR / "user_item_time_features_v1.parquet"
)

STEP3_LABEL_DIR = PROCESSED_DIR / "step3_labels"
STEP3_LABEL_DIR.mkdir(parents=True, exist_ok=True)

CANDIDATE_LABEL_PATH = (
    STEP3_LABEL_DIR / "candidate_purchase_labels_v1.parquet"
)

# 如果重新运行，只覆盖本步骤生成的同名文件
if PAIR_FEATURE_PATH.exists():
    PAIR_FEATURE_PATH.unlink()

if CANDIDATE_LABEL_PATH.exists():
    CANDIDATE_LABEL_PATH.unlink()

pair_feature_sql_path = PAIR_FEATURE_PATH.as_posix()
candidate_label_sql_path = CANDIDATE_LABEL_PATH.as_posix()

# 保存不含标签的用户—商品特征
con.execute(f"""
COPY (
    SELECT *
    FROM user_item_time_features
)
TO '{pair_feature_sql_path}'
(
    FORMAT PARQUET,
    COMPRESSION ZSTD
)
""")

# 单独保存标签
con.execute(f"""
COPY (
    SELECT
        pair.user_id,
        pair.item_id,

        CAST(
            COALESCE(lp.purchase_label, 0)
            AS UTINYINT
        ) AS purchase_label

    FROM user_item_time_features AS pair

    LEFT JOIN label_purchases AS lp
        ON pair.user_id = lp.user_id
       AND pair.item_id = lp.item_id
)
TO '{candidate_label_sql_path}'
(
    FORMAT PARQUET,
    COMPRESSION ZSTD
)
""")

# ============================================================
# 保存结果校验
# ============================================================

pair_feature_saved_check = con.execute(f"""
SELECT
    COUNT(*) AS feature_rows,
    COUNT(*) - COUNT(DISTINCT (user_id, item_id))
        AS duplicate_keys
FROM read_parquet('{pair_feature_sql_path}')
""").df()

candidate_label_saved_check = con.execute(f"""
SELECT
    COUNT(*) AS label_rows,

    SUM(purchase_label)
        AS positive_labels,

    COUNT(*) FILTER (
        WHERE purchase_label = 0
    ) AS negative_labels,

    100.0 * SUM(purchase_label) / COUNT(*)
        AS positive_rate_pct,

    COUNT(*) - COUNT(DISTINCT (user_id, item_id))
        AS duplicate_keys

FROM read_parquet('{candidate_label_sql_path}')
""").df()

save_summary = pd.DataFrame([
    {
        "file_type": "用户—商品特征",
        "file_name": PAIR_FEATURE_PATH.name,
        "file_size_mb": round(
            PAIR_FEATURE_PATH.stat().st_size / 1024 / 1024,
            2
        )
    },
    {
        "file_type": "独立购买标签",
        "file_name": CANDIDATE_LABEL_PATH.name,
        "file_size_mb": round(
            CANDIDATE_LABEL_PATH.stat().st_size / 1024 / 1024,
            2
        )
    }
])

print("===== 用户—商品特征保存校验 =====")
display(pair_feature_saved_check)

print("\n===== 独立标签保存校验 =====")
display(candidate_label_saved_check)

print("\n===== 文件保存结果 =====")
display(save_summary)

# 自动校验
assert int(
    pair_feature_saved_check.loc[0, "feature_rows"]
) == 4_546_934

assert int(
    pair_feature_saved_check.loc[0, "duplicate_keys"]
) == 0

assert int(
    candidate_label_saved_check.loc[0, "label_rows"]
) == 4_546_934

assert int(
    candidate_label_saved_check.loc[0, "positive_labels"]
) == 1_028

assert int(
    candidate_label_saved_check.loc[0, "negative_labels"]
) == 4_545_906

assert int(
    candidate_label_saved_check.loc[0, "duplicate_keys"]
) == 0

print("\n用户—商品特征与购买标签已分开保存，全部校验通过。")

===== 用户—商品特征保存校验 =====


,feature_rows,duplicate_keys
0,4546934,0



===== 独立标签保存校验 =====


,label_rows,positive_labels,negative_labels,positive_rate_pct,duplicate_keys
0,4546934,1028.0,4545906,0.022609,0



===== 文件保存结果 =====


,file_type,file_name,file_size_mb
0,用户—商品特征,user_item_time_features_v1.parquet,69.15
1,独立购买标签,candidate_purchase_labels_v1.parquet,26.37



用户—商品特征与购买标签已分开保存，全部校验通过。


In [14]:
# ============================================================
# 步骤三：检查五类特征的关联完整性
# ============================================================

feature_join_check = con.execute("""
SELECT
    COUNT(*) AS candidate_pairs,

    SUM(
        CASE WHEN u.user_id IS NULL
        THEN 1 ELSE 0 END
    ) AS missing_user_features,

    SUM(
        CASE WHEN i.item_id IS NULL
        THEN 1 ELSE 0 END
    ) AS missing_item_features,

    SUM(
        CASE WHEN c.item_category IS NULL
        THEN 1 ELSE 0 END
    ) AS missing_category_features,

    SUM(
        CASE WHEN uc.user_id IS NULL
        THEN 1 ELSE 0 END
    ) AS missing_user_category_features

FROM user_item_time_features AS pair

LEFT JOIN user_time_features AS u
    ON pair.user_id = u.user_id

LEFT JOIN item_time_features AS i
    ON pair.item_id = i.item_id

LEFT JOIN category_time_features AS c
    ON pair.item_category = c.item_category

LEFT JOIN user_category_features AS uc
    ON pair.user_id = uc.user_id
   AND pair.item_category = uc.item_category
""").df()

print("===== 五类特征关联完整性校验 =====")
display(feature_join_check)

assert int(
    feature_join_check.loc[0, "candidate_pairs"]
) == 4_546_934

assert int(
    feature_join_check.loc[0, "missing_user_features"]
) == 0

assert int(
    feature_join_check.loc[0, "missing_item_features"]
) == 0

assert int(
    feature_join_check.loc[0, "missing_category_features"]
) == 0

assert int(
    feature_join_check.loc[0, "missing_user_category_features"]
) == 0

print("\n五类特征可以完整关联，没有缺失的维度特征。")

===== 五类特征关联完整性校验 =====


,candidate_pairs,missing_user_features,missing_item_features,missing_category_features,missing_user_category_features
0,4546934,0.0,0.0,0.0,0.0



五类特征可以完整关联，没有缺失的维度特征。


In [15]:
# ============================================================
# 步骤三：特征质量与数据泄露检查
# ============================================================

feature_table_config = {
    "user_time_features": "user_last_action_time",
    "item_time_features": "item_last_action_time",
    "category_time_features": "category_last_action_time",
    "user_category_features": "user_category_last_action_time",
    "user_item_time_features": "pair_last_action_time"
}

quality_results = []

numeric_type_keywords = [
    "TINYINT",
    "SMALLINT",
    "INTEGER",
    "BIGINT",
    "HUGEINT",
    "UTINYINT",
    "USMALLINT",
    "UINTEGER",
    "UBIGINT",
    "FLOAT",
    "DOUBLE",
    "REAL",
    "DECIMAL"
]

floating_type_keywords = [
    "FLOAT",
    "DOUBLE",
    "REAL",
    "DECIMAL"
]

leakage_keywords = [
    "label",
    "target",
    "next_horizon",
    "future"
]

for table_name, last_time_column in feature_table_config.items():

    schema = con.execute(
        f"DESCRIBE {table_name}"
    ).df()

    all_columns = schema["column_name"].tolist()

    numeric_columns = []

    floating_columns = []

    for _, schema_row in schema.iterrows():
        column_name = schema_row["column_name"]
        column_type = str(schema_row["column_type"]).upper()

        if any(
            keyword in column_type
            for keyword in numeric_type_keywords
        ):
            numeric_columns.append(column_name)

        if any(
            keyword in column_type
            for keyword in floating_type_keywords
        ):
            floating_columns.append(column_name)

    # 检查全部字段的缺失值
    null_expression = " + ".join([
        f"""
        SUM(
            CASE WHEN "{column}" IS NULL
            THEN 1 ELSE 0 END
        )
        """
        for column in all_columns
    ])

    # 检查浮点数中的NaN和无穷值
    if floating_columns:
        nonfinite_expression = " + ".join([
            f"""
            SUM(
                CASE
                    WHEN "{column}" IS NOT NULL
                     AND NOT ISFINITE(
                         CAST("{column}" AS DOUBLE)
                     )
                    THEN 1
                    ELSE 0
                END
            )
            """
            for column in floating_columns
        ])
    else:
        nonfinite_expression = "0"

    # 当前特征均为计数、比例、排名或时间距离，
    # 因此不应该出现负值
    negative_expression = " + ".join([
        f"""
        SUM(
            CASE
                WHEN "{column}" < 0
                THEN 1
                ELSE 0
            END
        )
        """
        for column in numeric_columns
    ])

    table_quality = con.execute(f"""
    SELECT
        COUNT(*) AS row_count,

        ({null_expression})
            AS null_cells,

        ({nonfinite_expression})
            AS nonfinite_cells,

        ({negative_expression})
            AS negative_numeric_cells,

        MAX("{last_time_column}")
            AS latest_feature_time

    FROM {table_name}
    """).df()

    leakage_columns = [
        column
        for column in all_columns
        if any(
            keyword in column.lower()
            for keyword in leakage_keywords
        )
    ]

    quality_results.append({
        "feature_table": table_name,
        "row_count": int(
            table_quality.loc[0, "row_count"]
        ),
        "column_count": len(all_columns),
        "null_cells": int(
            table_quality.loc[0, "null_cells"]
        ),
        "nonfinite_cells": int(
            table_quality.loc[0, "nonfinite_cells"]
        ),
        "negative_numeric_cells": int(
            table_quality.loc[
                0,
                "negative_numeric_cells"
            ]
        ),
        "latest_feature_time":
            table_quality.loc[0, "latest_feature_time"],
        "leakage_columns":
            ", ".join(leakage_columns)
            if leakage_columns
            else "无"
    })

feature_quality_report = pd.DataFrame(quality_results)

print("===== 步骤三特征质量与泄露检查 =====")
display(feature_quality_report)

# 自动校验
assert feature_quality_report["null_cells"].sum() == 0
assert feature_quality_report["nonfinite_cells"].sum() == 0
assert feature_quality_report[
    "negative_numeric_cells"
].sum() == 0

assert (
    feature_quality_report["leakage_columns"] == "无"
).all()

assert (
    pd.to_datetime(
        feature_quality_report["latest_feature_time"]
    ) < CUTOFF_TIME
).all()

print(
    "\n特征质量检查通过："
    "无缺失值、无无穷值、无负值、"
    "无标签泄露，所有特征均来自观察期。"
)

===== 步骤三特征质量与泄露检查 =====


,feature_table,row_count,column_count,null_cells,nonfinite_cells,negative_numeric_cells,latest_feature_time,leakage_columns
0,user_time_features,9988,54,0,0,0,2025-12-17 23:00:00,无
1,item_time_features,2803933,33,0,0,0,2025-12-17 23:00:00,无
2,category_time_features,8875,54,0,0,0,2025-12-17 23:00:00,无
3,user_category_features,880669,27,0,0,0,2025-12-17 23:00:00,无
4,user_item_time_features,4546934,38,0,0,0,2025-12-17 23:00:00,无



特征质量检查通过：无缺失值、无无穷值、无负值、无标签泄露，所有特征均来自观察期。


In [16]:
# ============================================================
# 步骤三：生成特征工程报告与特征目录
# ============================================================

import re

REPORT_DIR = PROJECT_ROOT / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_CATALOG_PATH = (
    REPORT_DIR / "step3_feature_catalog.csv"
)

FEATURE_QUALITY_PATH = (
    REPORT_DIR / "step3_feature_quality_summary.csv"
)

COLD_START_PATH = (
    REPORT_DIR / "step3_cold_start_categories.csv"
)

STEP3_REPORT_PATH = (
    REPORT_DIR / "step3_feature_engineering_report.md"
)

# ------------------------------------------------------------
# 1. 生成特征字段目录
# ------------------------------------------------------------

feature_table_names = [
    "user_time_features",
    "item_time_features",
    "category_time_features",
    "user_category_features",
    "user_item_time_features"
]

table_name_cn = {
    "user_time_features": "用户多时间窗口特征",
    "item_time_features": "商品多时间窗口特征",
    "category_time_features": "品类多时间窗口特征",
    "user_category_features": "用户—品类偏好特征",
    "user_item_time_features": "用户—商品交互特征"
}

def get_feature_role(column_name):

    if column_name in [
        "user_id",
        "item_id",
        "item_category"
    ]:
        return "关联键"

    if (
        "first_action_time" in column_name
        or "last_action_time" in column_name
    ):
        return "时间审计字段"

    if "category_count" in column_name:
        return "数据质量字段"

    return "模型候选特征"


def get_feature_description(column_name):

    window_match = re.search(r"_(\d+)d$", column_name)

    if window_match:
        window_text = (
            f"预测截止日前最近"
            f"{window_match.group(1)}天"
        )
    else:
        window_text = "整个观察期"

    if column_name == "user_id":
        return "匿名化用户唯一编号"

    if column_name == "item_id":
        return "匿名化商品唯一编号"

    if column_name == "item_category":
        return "匿名化商品品类编号"

    if "first_action_time" in column_name:
        return "观察期内首次相关行为时间，仅用于审计"

    if "last_action_time" in column_name:
        return "观察期内最后一次相关行为时间，仅用于审计"

    if "recency_hours" in column_name:
        return "最后一次相关行为距离预测时点的小时数"

    if "total_actions" in column_name:
        return f"{window_text}内的总行为次数"

    if "view_count" in column_name:
        return f"{window_text}内的浏览次数"

    if "favorite_count" in column_name:
        return f"{window_text}内的收藏次数"

    if "cart_count" in column_name:
        return f"{window_text}内的加购次数"

    if "purchase_count" in column_name:
        return f"{window_text}内的历史购买次数"

    if "unique_users" in column_name:
        return f"{window_text}内的独立交互用户数"

    if "unique_buyers" in column_name:
        return f"{window_text}内的独立购买用户数"

    if "unique_items" in column_name:
        return f"{window_text}内的独立交互商品数"

    if "unique_categories" in column_name:
        return f"{window_text}内的独立交互品类数"

    if "active_days" in column_name:
        return f"{window_text}内发生行为的活跃天数"

    if "active_hours" in column_name:
        return f"{window_text}内发生行为的活跃小时数"

    if "repeat_view_count" in column_name:
        return "观察期内重复浏览次数，首次浏览不计入重复"

    if "purchase_action_pct" in column_name:
        return f"{window_text}内购买行为占全部行为的比例"

    if "high_intent_action_pct" in column_name:
        return f"{window_text}内收藏、加购及购买行为占比"

    if "action_share" in column_name:
        return f"{window_text}内该品类行为占用户全部行为的比例"

    if "preference_rank" in column_name:
        return "该品类在用户历史偏好品类中的排序"

    if "category_count" in column_name:
        return "同一商品关联的品类数量，用于一致性检查"

    return "基于观察期历史行为构建的候选特征"


catalog_rows = []

for table_name in feature_table_names:

    schema = con.execute(
        f"DESCRIBE {table_name}"
    ).df()

    for _, schema_row in schema.iterrows():

        column_name = schema_row["column_name"]

        catalog_rows.append({
            "feature_table": table_name,
            "feature_table_cn":
                table_name_cn[table_name],
            "feature_name": column_name,
            "data_type":
                schema_row["column_type"],
            "feature_role":
                get_feature_role(column_name),
            "feature_description":
                get_feature_description(column_name),
            "feature_cutoff_time":
                "2025-12-18 00:00:00之前"
        })

feature_catalog = pd.DataFrame(catalog_rows)

feature_catalog.to_csv(
    FEATURE_CATALOG_PATH,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 2. 保存质量摘要与冷启动明细
# ------------------------------------------------------------

feature_quality_report.to_csv(
    FEATURE_QUALITY_PATH,
    index=False,
    encoding="utf-8-sig"
)

cold_start_categories.to_csv(
    COLD_START_PATH,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 3. 生成特征表汇总
# ------------------------------------------------------------

feature_output_paths = {
    "user_time_features":
        STEP3_FEATURE_DIR / "user_time_features_v1.parquet",

    "item_time_features":
        STEP3_FEATURE_DIR / "item_time_features_v1.parquet",

    "category_time_features":
        STEP3_FEATURE_DIR / "category_time_features_v1.parquet",

    "user_category_features":
        STEP3_FEATURE_DIR / "user_category_features_v1.parquet",

    "user_item_time_features":
        STEP3_FEATURE_DIR / "user_item_time_features_v1.parquet"
}

feature_summary_rows = []

for table_name in feature_table_names:

    quality_row = feature_quality_report[
        feature_quality_report["feature_table"]
        == table_name
    ].iloc[0]

    output_path = feature_output_paths[table_name]

    feature_summary_rows.append({
        "特征表": table_name,
        "业务含义": table_name_cn[table_name],
        "行数": int(quality_row["row_count"]),
        "字段数": int(quality_row["column_count"]),
        "文件大小MB": round(
            output_path.stat().st_size / 1024 / 1024,
            2
        )
    })

feature_summary_df = pd.DataFrame(feature_summary_rows)

# ------------------------------------------------------------
# 4. Markdown表格辅助函数
# ------------------------------------------------------------

def dataframe_to_markdown(dataframe):

    columns = dataframe.columns.tolist()

    header = (
        "| " + " | ".join(columns) + " |"
    )

    separator = (
        "|"
        + "|".join(["---"] * len(columns))
        + "|"
    )

    rows = []

    for _, dataframe_row in dataframe.iterrows():

        values = []

        for column in columns:
            value = str(dataframe_row[column])
            value = value.replace("|", "\\|")
            values.append(value)

        rows.append(
            "| " + " | ".join(values) + " |"
        )

    return "\n".join(
        [header, separator] + rows
    )

feature_summary_markdown = dataframe_to_markdown(
    feature_summary_df
)

quality_markdown = dataframe_to_markdown(
    feature_quality_report
)

# ------------------------------------------------------------
# 5. 生成步骤三Markdown报告
# ------------------------------------------------------------

step3_report_text = f"""# 淘宝用户行为购买预测：步骤三特征工程报告

## 1. 特征工程范围

本步骤基于标准化用户行为数据构建购买预测所需的历史行为特征。

- 观察期开始：2025-11-18 00:00:00
- 特征截止时间：2025-12-18 00:00:00
- 标签期：2025-12-18 00:00:00至2025-12-19 00:00:00
- 观察期行为记录：11,881,309条
- 标签期行为记录：375,597条
- 标签期购买行为：3,586次
- 标签期购买用户—商品组合：3,151个

所有模型候选特征均严格使用特征截止时间之前的数据。

## 2. 特征体系

本步骤构建了五类特征：

1. 用户多时间窗口特征：反映用户活跃度、行为偏好、购买倾向和最近活跃时间；
2. 商品多时间窗口特征：反映商品短期热度、历史热度和购买表现；
3. 品类多时间窗口特征：反映品类流量、用户规模和购买表现；
4. 用户—品类偏好特征：反映用户对特定品类的历史兴趣强度和偏好排名；
5. 用户—商品交互特征：反映用户对具体商品的浏览、收藏、加购、购买、重复浏览及行为时效性。

用户和品类采用1、3、7、14、30天窗口；
商品采用1、7、30天窗口；
用户—商品采用1、3、7、14、30天窗口。

## 3. 特征表汇总

{feature_summary_markdown}

## 4. 重复行为处理

本步骤没有删除重复浏览、收藏、加购或购买行为。

重复行为被转化为行为次数、重复浏览次数、活跃天数、活跃小时数和行为占比等特征，用于反映用户兴趣强度和购买意愿。

## 5. 冷启动品类检查

完整数据包含8,916个品类，观察期包含8,875个品类。

差异的41个品类仅在标签日出现，共涉及：

- 109条行为；
- 39名用户；
- 51件商品；
- 0次购买。

因此这41个冷启动品类不影响本次购买正样本，但没有使用其标签日行为生成历史特征。

## 6. 候选用户—商品组合

第一版候选集合定义为观察期内出现过的用户—商品组合：

- 候选组合数：4,546,934；
- 标签期正样本：1,028；
- 负样本：4,545,906；
- 正样本率：0.022609%；
- 对全部3,151个标签期购买组合的覆盖率：32.6246%。

该候选集合适用于历史兴趣商品的购买预测。

新商品和历史未交互商品的召回属于后续候选扩展任务，不能通过使用标签日信息人为补齐。

## 7. 特征关联与质量校验

五类特征可通过用户、商品和品类关联键完整连接，4,546,934个候选组合均不存在维度特征缺失。

{quality_markdown}

质量检查结果：

- 缺失值：0；
- NaN或无穷值：0；
- 不合理负值：0；
- 重复候选主键：0；
- 标签字段混入特征：0；
- 特征最晚时间：2025-12-17 23:00:00。

## 8. 数据存储原则

为避免将大量用户级、商品级和品类级特征在454万条候选记录中重复存储，本步骤采用分层特征表设计。

正式建模时先确定训练样本，再通过关联键读取所需特征，可以显著减少内存和磁盘占用。

标签文件与特征文件独立保存，避免误将标签作为输入特征。

## 9. 步骤三结论

步骤三已完成用户、商品、品类、用户—品类和用户—商品五类历史特征构建。

全部特征均来自观察期，未使用标签期信息，特征表记录数、主键、时间范围和关联完整性均通过校验。

当前产出已经具备进入下一步骤进行样本划分、类别不平衡处理和模型训练的条件。
"""

STEP3_REPORT_PATH.write_text(
    step3_report_text,
    encoding="utf-8"
)

# ------------------------------------------------------------
# 6. 最终校验
# ------------------------------------------------------------

assert FEATURE_CATALOG_PATH.exists()
assert FEATURE_QUALITY_PATH.exists()
assert COLD_START_PATH.exists()
assert STEP3_REPORT_PATH.exists()
assert len(feature_catalog) > 0

print("===== 步骤三报告生成结果 =====")
print("特征目录：", FEATURE_CATALOG_PATH)
print("质量摘要：", FEATURE_QUALITY_PATH)
print("冷启动明细：", COLD_START_PATH)
print("步骤三报告：", STEP3_REPORT_PATH)

print("\n特征目录字段记录数：", len(feature_catalog))

print("\n===== 报告预览 =====")
print(
    "\n".join(
        step3_report_text.splitlines()[:35]
    )
)

===== 步骤三报告生成结果 =====
特征目录： /Users/nyx/Desktop/user-behavior-analysis/reports/step3_feature_catalog.csv
质量摘要： /Users/nyx/Desktop/user-behavior-analysis/reports/step3_feature_quality_summary.csv
冷启动明细： /Users/nyx/Desktop/user-behavior-analysis/reports/step3_cold_start_categories.csv
步骤三报告： /Users/nyx/Desktop/user-behavior-analysis/reports/step3_feature_engineering_report.md

特征目录字段记录数： 206

===== 报告预览 =====
# 淘宝用户行为购买预测：步骤三特征工程报告

## 1. 特征工程范围

本步骤基于标准化用户行为数据构建购买预测所需的历史行为特征。

- 观察期开始：2025-11-18 00:00:00
- 特征截止时间：2025-12-18 00:00:00
- 标签期：2025-12-18 00:00:00至2025-12-19 00:00:00
- 观察期行为记录：11,881,309条
- 标签期行为记录：375,597条
- 标签期购买行为：3,586次
- 标签期购买用户—商品组合：3,151个

所有模型候选特征均严格使用特征截止时间之前的数据。

## 2. 特征体系

本步骤构建了五类特征：

1. 用户多时间窗口特征：反映用户活跃度、行为偏好、购买倾向和最近活跃时间；
2. 商品多时间窗口特征：反映商品短期热度、历史热度和购买表现；
3. 品类多时间窗口特征：反映品类流量、用户规模和购买表现；
4. 用户—品类偏好特征：反映用户对特定品类的历史兴趣强度和偏好排名；
5. 用户—商品交互特征：反映用户对具体商品的浏览、收藏、加购、购买、重复浏览及行为时效性。

用户和品类采用1、3、7、14、30天窗口；
商品采用1、7、30天窗口；
用户—商品采用1、3、7、14、30天窗口。

## 3. 特征表汇总

| 特征表 | 业务含义 | 行数 | 字段

In [17]:
# 修正步骤三报告中的时间窗口说明

report_text = STEP3_REPORT_PATH.read_text(
    encoding="utf-8"
)

old_text = """用户和品类采用1、3、7、14、30天窗口；
商品采用1、7、30天窗口；
用户—商品采用1、3、7、14、30天窗口。"""

new_text = """用户和品类采用1、3、7、14、30天窗口；
商品采用1、7、30天窗口；
用户—品类采用1、7、30天窗口；
用户—商品采用1、3、7、14、30天窗口。"""

assert old_text in report_text

report_text = report_text.replace(
    old_text,
    new_text
)

STEP3_REPORT_PATH.write_text(
    report_text,
    encoding="utf-8"
)

print("步骤三报告时间窗口说明已修正。")

步骤三报告时间窗口说明已修正。
